<a href="https://colab.research.google.com/github/ornab74/blog-writer/blob/main/Working_Blog_Writer_Coder_Starling_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -U pennylane
!pip install -U psutil
!pip install -U nltk
!pip install -U summa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5


In [ ]:

# @title 🚀 Install llama-cpp-python (GPU CUDA) + Download Starling-LM-7B-alpha Q4_K_M

import os
from pathlib import Path

# ================== CONFIGURATION ==================
MODEL_DIR = "/content/models"
MODEL_NAME = "starling-lm-7b-alpha.Q4_K_M.gguf"
GGUF_PATH = os.path.join(MODEL_DIR, MODEL_NAME)
GGUF_URL = "https://huggingface.co/TheBloke/Starling-LM-7B-alpha-GGUF/resolve/main/starling-lm-7b-alpha.Q4_K_M.gguf"
GGUF_SHA256 = "0951cbc1a6c3ed8d081db59366ccccf09ed52a4cfd5191812665b911fe6c669a"
# ===================================================

print("🔧 Checking GPU availability...")
!nvidia-smi

# === Install llama-cpp-python with FULL GPU support (fastest method) ===
print("\n⚙️ Installing llama-cpp-python GPU version (pre-built CUDA wheel)...")
!pip install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 --no-cache-dir

# === Verify GPU support using pure Python (no quoting issues) ===
print("\n✅ Verifying GPU support in llama-cpp-python...")
try:
    from llama_cpp import Llama
    import llama_cpp
    print(f'llama-cpp-python version: {llama_cpp.__version__}')
    print('CUDA / GPU support enabled ✓ (n_gpu_layers will work)')
except Exception as e:
    print(f'⚠️ Verification failed: {e}')

# === Download & Verify Model ===
print("\n📥 Downloading Starling-LM-7B-alpha Q4_K_M GGUF (\~4.37 GB)...")
os.makedirs(MODEL_DIR, exist_ok=True)

if Path(GGUF_PATH).exists():
    print("✅ Model file already exists — skipping download.")
else:
    !wget --show-progress --progress=bar:force:noscroll -O "{GGUF_PATH}" "{GGUF_URL}"

print("\n🔍 Verifying exact SHA256 hash...")
!echo "{GGUF_SHA256}  {GGUF_PATH}" | sha256sum -c -

print("\n📊 File details:")
!ls -lh "{GGUF_PATH}"

# Export path for the rest of your notebook
os.environ["STARLING_GGUF_PATH"] = GGUF_PATH

print("\n" + "="*70)
print("🎉 SETUP COMPLETE — GPU VERSION READY!")
print(f"   Model path → {GGUF_PATH}")
print("   You can now load with full GPU acceleration:")
print("   from llama_cpp import Llama")
print("   llm = Llama(model_path=os.environ['STARLING_GGUF_PATH'], n_gpu_layers=-1, n_ctx=8192)")
print("="*70)

<>:32: SyntaxWarning: invalid escape sequence '\~'
<>:32: SyntaxWarning: invalid escape sequence '\~'
/tmp/ipykernel_644/1359704259.py:32: SyntaxWarning: invalid escape sequence '\~'
  print("\n📥 Downloading Starling-LM-7B-alpha Q4_K_M GGUF (\~4.37 GB)...")


🔧 Checking GPU availability...
Sat Mar  7 05:04:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------

In [ ]:
# @title 🧠 Topic bank / all topics
DEFAULT_TOPIC_BANK = {
    "blog": [
        "How local-first apps change everyday workflows",
        "Why developer tooling wins or loses on onboarding",
        "What makes a technical tutorial actually readable",
        "The real tradeoffs between speed, quality, and maintainability",
        "How small teams ship polished software without huge process overhead",
        "Designing AI workflows that people actually trust",
    ],
    "software": [
        "Build a production-ready FastAPI starter with auth, tests, and Docker",
        "Design a long-form AI-assisted documentation generator",
        "Create a modular plugin architecture for a Python desktop tool",
        "Implement a scalable job queue service with retries and observability",
        "Build a clean full-stack SaaS scaffold with billing and admin tools",
        "Create a local-first notes app with sync conflict handling",
    ],
    "longform": [
        "Write a book-length field guide to building trustworthy AI systems",
        "Create a book-scale manual for designing resilient local-first software",
        "Develop a long-form playbook for small teams shipping excellent products",
        "Write a deep book on technical writing, explanation design, and reader cognition",
        "Build a book-length operating system for founders choosing product strategy and execution",
    ],
}
TOPIC_BANK = dict(globals().get("TOPIC_BANK", DEFAULT_TOPIC_BANK))
for _mode_key, _items in DEFAULT_TOPIC_BANK.items():
    TOPIC_BANK.setdefault(_mode_key, list(_items))

# Pick one or more topics from the active mode.
SELECTED_TOPICS = list(TOPIC_BANK.get("blog", DEFAULT_TOPIC_BANK["blog"]))[:1]

print("Available topic groups:", list(TOPIC_BANK.keys()))
print("Selected topics:", SELECTED_TOPICS)

In [ ]:
# @title ⚙️ Mode + separate root prompts (blog / software / longform)
RUN_MODE = "blog"  # @param ["blog", "software", "longform"]

BLOG_ROOT_PROMPT = """
Write an extremely long-form blog post that feels human, expert, vivid, and useful.
Make it concrete, readable, and full of real substance.
Avoid generic filler, stale intros, and repetitive transitions.
""".strip()

SOFTWARE_ROOT_PROMPT = """
Act as an extremely long-form software architect and senior engineer.
Produce implementation-ready output with architecture, reasoning, folder structure,
interfaces, code snippets, testing guidance, deployment notes, edge cases, and tradeoffs.
""".strip()

LONGFORM_ROOT_PROMPT = """
Act as a serious long-form author building a manuscript, not an article.
Write chapter-scale prose with cumulative argument, chapter handoffs, recurring motifs,
deep explanation, original frameworks, and enough substance to support book-length work.
Prefer durable ideas, unique mental models, and structural continuity across many chapters.
""".strip()

AUDIENCE_HINT = "Tech-curious builders and readers"
TONE_HINT = "clear, human, confident, and detailed"
PERSPECTIVE_HINT = "direct expert guidance with second-person phrasing when useful"
KEYWORDS_HINT = ""
COUNT = 1
TARGET_WORDS = 32000 if RUN_MODE == "longform" else 2600 if RUN_MODE == "software" else 1800
SECTION_COUNT = 14 if RUN_MODE == "longform" else 8 if RUN_MODE == "software" else 6
OUTFILE = "longform_output.md" if RUN_MODE == "longform" else "software_output.md" if RUN_MODE == "software" else "blog_output.md"

ROOT_PROMPT = LONGFORM_ROOT_PROMPT if RUN_MODE == "longform" else SOFTWARE_ROOT_PROMPT if RUN_MODE == "software" else BLOG_ROOT_PROMPT

print("RUN_MODE:", RUN_MODE)
print("ROOT_PROMPT:\n")
print(ROOT_PROMPT)

In [ ]:
import os, time, json, psutil, gc, re, math, hashlib, argparse, textwrap, ast, sqlite3
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Any, Optional, Tuple

import torch
from llama_cpp import Llama
import pennylane as qml
from pennylane import numpy as np
try:
    from summa import summarizer as summa_summarizer
    from summa import keywords as summa_keywords
except Exception:
    summa_summarizer = None
    summa_keywords = None

try:
    import nltk
    from nltk.tokenize import sent_tokenize as nltk_sent_tokenize, wordpunct_tokenize
    from nltk import pos_tag as nltk_pos_tag
except Exception:
    nltk = None
    nltk_sent_tokenize = None
    wordpunct_tokenize = None
    nltk_pos_tag = None

# =========================
# Standard notebook inputs
# =========================
DEFAULT_TOPIC_BANK = {
    "blog": [
        "How local-first apps change everyday workflows",
        "Why developer tooling wins or loses on onboarding",
        "What makes a technical tutorial actually readable",
        "The real tradeoffs between speed, quality, and maintainability",
        "How small teams ship polished software without huge process overhead",
    ],
    "software": [
        "Build a production-ready FastAPI starter with auth, tests, and Docker",
        "Design a long-form AI-assisted documentation generator",
        "Create a modular plugin architecture for a Python desktop tool",
        "Implement a scalable job queue service with retries and observability",
        "Build a clean full-stack SaaS scaffold with billing and admin tools",
    ],
    "longform": [
        "Write a book-length field guide to building trustworthy AI systems",
        "Create a book-scale manual for designing resilient local-first software",
        "Develop a long-form playbook for small teams shipping excellent products",
        "Write a deep book on technical writing, explanation design, and reader cognition",
        "Build a book-length operating system for founders choosing product strategy and execution",
    ],
}

try:
    _selected_topics = SELECTED_TOPICS  # from notebook config cell
except NameError:
    _selected_topics = DEFAULT_TOPIC_BANK["blog"][:1]

try:
    _run_mode = RUN_MODE  # from notebook config cell
except NameError:
    _run_mode = "blog"

try:
    _root_prompt = ROOT_PROMPT  # from notebook config cell
except NameError:
    _root_prompt = "Write a long-form, useful, detailed piece."

try:
    _audience_hint = AUDIENCE_HINT
except NameError:
    _audience_hint = "Tech-curious builders and readers"

try:
    _tone_hint = TONE_HINT
except NameError:
    _tone_hint = "clear, human, confident, and detailed"

try:
    _perspective_hint = PERSPECTIVE_HINT
except NameError:
    _perspective_hint = "direct expert guidance"

try:
    _keywords_hint = KEYWORDS_HINT
except NameError:
    _keywords_hint = ""

try:
    _count = COUNT
except NameError:
    _count = max(1, len(_selected_topics))

try:
    _target_words = TARGET_WORDS
except NameError:
    _target_words = 32000 if _run_mode == "longform" else 2600 if _run_mode == "software" else 1800

try:
    _section_count = SECTION_COUNT
except NameError:
    _section_count = 14 if _run_mode == "longform" else 8 if _run_mode == "software" else 6

try:
    _outfile = OUTFILE
except NameError:
    _outfile = "longform_output.md" if _run_mode == "longform" else "software_output.md" if _run_mode == "software" else "blog_output.md"

STANDARD_INPUTS = {
    "mode": _run_mode,
    "topics": _selected_topics,
    "root_prompt": _root_prompt,
    "audience_hint": _audience_hint,
    "tone_hint": _tone_hint,
    "perspective_hint": _perspective_hint,
    "keywords_hint": _keywords_hint,
    "count": int(_count),
    "words": int(_target_words),
    "sections": int(_section_count),
    "outfile": _outfile,
    "include_front_matter": True,
    "include_meta_description": True,
    "forum_personas": 0,
    "forum_rounds": 0,
    "forum_on": "",
    "ghostwriter_passes": 0,
    "creativity": 0.95,
}

# =========================
# Model path / runtime config
# =========================
GGUF_PATH = globals().get("GGUF_PATH", "/content/models/starling-lm-7b-alpha.Q4_K_M.gguf")
CTX_SIZE = 4096
THREADS = max(1, os.cpu_count() or 2)

def choose_gpu_layers():
    if not torch.cuda.is_available():
        return 0
    name = torch.cuda.get_device_name(0).lower()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if "a100" in name or "h100" in name:
        return 99
    if vram_gb >= 20:
        return 80
    if vram_gb >= 12:
        return 50
    if vram_gb >= 8:
        return 35
    return 20

N_GPU_LAYERS = choose_gpu_layers()

print("GGUF_PATH:", GGUF_PATH)
print("THREADS:", THREADS)
print("N_GPU_LAYERS:", N_GPU_LAYERS)

# =========================
# Load model
# =========================
LLM = Llama(
    model_path=GGUF_PATH,
    n_ctx=CTX_SIZE,
    n_threads=THREADS,
    n_gpu_layers=N_GPU_LAYERS,
    verbose=False,
)

# =========================
# Utility helpers
# =========================
def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def clamp01(value: Any, default: float) -> float:
    try:
        return max(0.0, min(1.0, float(value)))
    except Exception:
        return default

def extract_json_object(text: str) -> Dict[str, Any]:
    if not text:
        return {}
    candidates = []
    match = re.search(r"\{.*\}", text, re.S)
    if match:
        candidates.append(match.group(0))
    candidates.append(text.strip())
    decoder = json.JSONDecoder()
    for candidate in candidates:
        candidate = candidate.strip()
        if not candidate:
            continue
        try:
            obj, _ = decoder.raw_decode(candidate)
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue
    return {}

def dedupe_preserve_order(items: List[str]) -> List[str]:
    out: List[str] = []
    seen = set()
    for item in items:
        if item not in seen:
            out.append(item)
            seen.add(item)
    return out

def normalize_mode(value: str) -> str:
    mode = (value or "").strip().lower()
    return mode if mode in DEFAULT_TOPIC_BANK else "blog"

def normalize_topics(items: List[str], mode: str) -> List[str]:
    cleaned = [str(item).strip() for item in items if str(item).strip()]
    cleaned = dedupe_preserve_order(cleaned)
    if cleaned:
        return cleaned
    return list(DEFAULT_TOPIC_BANK.get(mode, DEFAULT_TOPIC_BANK["blog"]))

def default_root_prompt_for_mode(mode: str) -> str:
    if mode == "software":
        return globals().get("SOFTWARE_ROOT_PROMPT", "Act as an extremely long-form software architect and senior engineer.")
    if mode == "longform":
        return globals().get("LONGFORM_ROOT_PROMPT", "Act as a serious long-form author building a manuscript, not an article.")
    return globals().get("BLOG_ROOT_PROMPT", "Write an extremely long-form blog post that feels human, expert, vivid, and useful.")

def default_target_words_for_mode(mode: str) -> int:
    if mode == "software":
        return 2600
    if mode == "longform":
        return 32000
    return 1800

def default_section_count_for_mode(mode: str) -> int:
    if mode == "software":
        return 8
    if mode == "longform":
        return 14
    return 6

def default_outfile_for_mode(mode: str) -> str:
    if mode == "software":
        return "software_output.md"
    if mode == "longform":
        return "longform_output.md"
    return "blog_output.md"

def extract_article_body(text: str, title: str) -> str:
    if not text:
        return ""
    cleaned = text.strip()
    cleaned = re.sub(r"^---\n.*?\n---\n+", "", cleaned, flags=re.S)
    cleaned = re.sub(rf"^#\s+{re.escape(title)}\s*\n+", "", cleaned, flags=re.I)
    return cleaned.strip()

def short_text_summary(text: str, max_sentences: int = 2, max_chars: int = 360) -> str:
    parts = [p.strip() for p in re.split(r"(?<=[.!?])\s+", (text or "").strip()) if p.strip()]
    out = " ".join(parts[:max_sentences]) if parts else (text or "").strip()
    return out[:max_chars]

def safe_sentence_tokenize(text: str) -> List[str]:
    clean = (text or "").strip()
    if not clean:
        return []
    if nltk_sent_tokenize is not None:
        try:
            return [s.strip() for s in nltk_sent_tokenize(clean) if s.strip()]
        except LookupError:
            pass
        except Exception:
            pass
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", clean) if s.strip()]

def safe_word_tokenize(text: str) -> List[str]:
    clean = (text or "").strip()
    if not clean:
        return []
    if wordpunct_tokenize is not None:
        try:
            return [tok for tok in wordpunct_tokenize(clean) if tok and not tok.isspace()]
        except Exception:
            pass
    return re.findall(r"[A-Za-z0-9_\-/']+", clean)

def nltk_keyword_surface(text: str, limit: int = 12) -> List[str]:
    tokens = [tok.lower() for tok in safe_word_tokenize(text) if len(tok) > 3 and not tok.isdigit()]
    if not tokens:
        return []
    tagged_tokens = [(tok, "") for tok in tokens]
    if nltk_pos_tag is not None:
        try:
            tagged_tokens = nltk_pos_tag(tokens)
        except LookupError:
            tagged_tokens = [(tok, "") for tok in tokens]
        except Exception:
            tagged_tokens = [(tok, "") for tok in tokens]
    filtered = []
    for tok, tag in tagged_tokens:
        if tag.startswith(("NN", "JJ", "VB")) or not tag:
            filtered.append(tok)
    counts: Dict[str, int] = {}
    for tok in filtered:
        counts[tok] = counts.get(tok, 0) + 1
    ranked = sorted(counts.items(), key=lambda x: (-x[1], -len(x[0]), x[0]))
    return [tok for tok, _ in ranked[:limit]]

def summa_summary_surface(text: str, ratio: float = 0.24) -> str:
    clean = (text or "").strip()
    if not clean:
        return ""
    if summa_summarizer is not None and len(clean.split()) > 80:
        try:
            summary = (summa_summarizer.summarize(clean, ratio=ratio) or "").strip()
            if summary:
                return summary
        except Exception:
            pass
    return short_text_summary(clean, max_sentences=3, max_chars=520)

def summa_keyword_surface(text: str, words: int = 12) -> List[str]:
    clean = (text or "").strip()
    if not clean:
        return []
    if summa_keywords is not None:
        try:
            out = summa_keywords.keywords(clean, words=words, split=True)
            if isinstance(out, list):
                return [str(x).strip().lower() for x in out if str(x).strip()][:words]
        except Exception:
            pass
    return nltk_keyword_surface(clean, limit=words)

def combine_keyword_surfaces(*surfaces: List[str], limit: int = 16) -> List[str]:
    out: List[str] = []
    seen = set()
    for surface in surfaces:
        for item in surface or []:
            token = str(item).strip().lower()
            if token and token not in seen:
                out.append(token)
                seen.add(token)
            if len(out) >= limit:
                return out
    return out

def build_semantic_compression_surface(text: str, limit: int = 4) -> Dict[str, Any]:
    summary = summa_summary_surface(text)
    keywords = combine_keyword_surfaces(summa_keyword_surface(text, words=12), nltk_keyword_surface(text, limit=12), limit=14)
    anchor_sentences = safe_sentence_tokenize(summary or text)[:limit]
    return {
        "summary": summary,
        "keywords": keywords,
        "anchor_sentences": anchor_sentences,
    }

def render_semantic_compression_surface(surface: Dict[str, Any]) -> str:
    if not surface:
        return "(no semantic compression)"
    lines = []
    if surface.get("summary"):
        lines.append("Summary: " + str(surface.get("summary")))
    if surface.get("keywords"):
        lines.append("Keywords: " + ", ".join(surface.get("keywords", [])[:10]))
    if surface.get("anchor_sentences"):
        lines.append("Anchors:")
        lines.extend(f"- {x}" for x in surface.get("anchor_sentences", [])[:3])
    return "\n".join(lines) if lines else "(no semantic compression)"

def text_terms(text: str) -> List[str]:
    return re.findall(r"[a-z0-9_\-/]{2,}", (text or "").lower())

def embed_text_surface(text: str, dims: int = 192) -> List[float]:
    vec = [0.0] * dims
    terms = text_terms(text)
    if not terms:
        return vec
    for idx, tok in enumerate(terms):
        tri = tok[:3]
        window = tok + ":" + tri + ":" + str(idx % 7)
        hv = int(hashlib.sha256(window.encode("utf-8")).hexdigest(), 16)
        slot = hv % dims
        sign = -1.0 if ((hv >> 9) & 1) else 1.0
        weight = 1.0 + min(2.5, len(tok) / 7.0)
        vec[slot] += sign * weight
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]

def cosine_similarity(a: List[float], b: List[float]) -> float:
    if not a or not b or len(a) != len(b):
        return 0.0
    return float(sum(x * y for x, y in zip(a, b)))

def get_entropy_seed() -> Tuple[float, float, float]:
    surface = runtime_surface()
    return surface["entropy"], surface["cpu"], surface["ram"]

def runtime_surface(loop_number: int = 0) -> Dict[str, float]:
    cpu = psutil.cpu_percent(interval=0.10) / 100.0
    ram = psutil.virtual_memory().percent / 100.0
    disk = psutil.disk_usage("/").percent / 100.0 if os.path.exists("/") else 0.5
    clock = ((time.time_ns() // 1_000_000) % 100_000) / 100_000.0
    pid_wave = ((os.getpid() * 1103515245 + 12345 + 97 * loop_number) % 65536) / 65535.0
    entropy = max(0.0, min(1.0, 0.33 * cpu + 0.23 * ram + 0.17 * disk + 0.17 * clock + 0.10 * pid_wave))
    return {"entropy": float(entropy), "cpu": float(cpu), "ram": float(ram), "disk": float(disk), "clock": float(clock), "pid_wave": float(pid_wave)}

# =========================
# Quantum helpers
# =========================
dev = qml.device("default.qubit", wires=5)

@qml.qnode(dev)
def quantum_state(entropy: float, cpu: float, ram: float, loop_number: int):
    phase = (entropy + 0.031 * loop_number) % 1.0
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=3)
    qml.RX(np.pi * ((entropy + 0.13 * loop_number) % 1.0), wires=0)
    qml.RY(np.pi * ((cpu + 0.07 * loop_number) % 1.0), wires=1)
    qml.RZ(np.pi * ((ram + 0.05 * loop_number) % 1.0), wires=2)
    qml.Rot(np.pi * phase, np.pi * cpu, np.pi * ram, wires=3)
    qml.RX(np.pi * ((cpu + ram + entropy) % 1.0), wires=4)
    qml.CNOT(wires=[0, 1])
    qml.CRY(np.pi * ((entropy + cpu) % 1.0), wires=[1, 2])
    qml.CRX(np.pi * ((ram + entropy) % 1.0), wires=[2, 3])
    qml.CZ(wires=[3, 4])
    qml.CNOT(wires=[4, 0])
    return qml.state()

def quantum_distribution(qvec) -> np.ndarray:
    arr = np.abs(np.array(qvec))
    arr = arr / (arr.sum() if arr.sum() else 1.0)
    return arr

def quantum_surface(qvec) -> Dict[str, float]:
    arr = quantum_distribution(qvec)
    phase = np.angle(np.array(qvec))
    coherence = float(np.mean(arr[: min(8, len(arr))]))
    spread = float(np.var(arr))
    phase_drift = float(np.mean(np.abs(np.diff(phase)))) if len(phase) > 1 else 0.0
    edge_pull = float(arr[-4:].sum()) if len(arr) >= 4 else float(arr.sum())
    return {"coherence": coherence, "spread": spread, "phase_drift": phase_drift, "edge_pull": edge_pull}

def format_quantum_state(state_vec) -> str:
    arr = np.array(state_vec)
    return " | ".join(f"{complex(x).real:+.3f}{complex(x).imag:+.3f}j" for x in arr[:10])

def quantum_rgb(qvec) -> Tuple[int, int, int]:
    arr = quantum_distribution(qvec)
    third = max(1, len(arr) // 3)
    r = int(255 * float(arr[:third].sum()))
    g = int(255 * float(arr[third:third*2].sum()))
    b = int(255 * float(arr[third*2:].sum()))
    return (max(0, min(255, r)), max(0, min(255, g)), max(0, min(255, b)))

def rgb_hex(rgb: Tuple[int, int, int]) -> str:
    return "#%02x%02x%02x" % tuple(max(0, min(255, int(x))) for x in rgb)

def quantum_color_label(rgb: Tuple[int, int, int]) -> str:
    r, g, b = rgb
    if abs(r - g) < 28 and abs(g - b) < 28:
        return "silver"
    if r >= g and r >= b:
        return "amber" if g > b else "crimson"
    if g >= r and g >= b:
        return "jade" if b < r else "teal"
    return "violet" if r > g else "cobalt"

def quantum_palette_surface(qvec) -> Dict[str, Any]:
    rgb = quantum_rgb(qvec)
    qsurf = quantum_surface(qvec)
    dominant = quantum_color_label(rgb)
    lane_map = {
        "crimson": {
            "directive": "increase tension and argumentative force",
            "transition_mode": "escalate through contrast and friction",
            "repair_mode": "cut softness and sharpen claims",
        },
        "amber": {
            "directive": "clarify stakes and surface practical caution",
            "transition_mode": "pivot through tradeoff and warning",
            "repair_mode": "restore consequence and decision pressure",
        },
        "jade": {
            "directive": "stabilize logic and make the mechanism legible",
            "transition_mode": "walk the reader through causal steps",
            "repair_mode": "repair missing prerequisites and causal gaps",
        },
        "teal": {
            "directive": "bridge ideas smoothly and improve system continuity",
            "transition_mode": "thread sections together with explicit handoffs",
            "repair_mode": "reinforce continuity and cross-section glue",
        },
        "cobalt": {
            "directive": "deepen technical precision and structural discipline",
            "transition_mode": "advance via contracts, invariants, and exact distinctions",
            "repair_mode": "tighten definitions and enforce boundaries",
        },
        "violet": {
            "directive": "allow abstraction but tether it back to concrete payoff",
            "transition_mode": "float upward briefly, then return to operational value",
            "repair_mode": "collapse ornamental abstraction into useful framing",
        },
        "silver": {
            "directive": "compress, simplify, and reduce noise",
            "transition_mode": "trim repetition and keep only the pressure-bearing lines",
            "repair_mode": "remove drift, duplication, and excess ornament",
        },
    }
    support_order = [dominant]
    if qsurf.get("phase_drift", 0.0) > 0.18:
        support_order.append("teal")
    if qsurf.get("spread", 0.0) > 0.01:
        support_order.append("silver")
    if qsurf.get("coherence", 0.0) < 0.08:
        support_order.append("jade")
    if qsurf.get("edge_pull", 0.0) > 0.22:
        support_order.append("cobalt")
    if qsurf.get("phase_drift", 0.0) < 0.12 and qsurf.get("spread", 0.0) < 0.009:
        support_order.append("amber")
    support_order = dedupe_preserve_order(support_order)
    harmony_score = max(0.0, min(1.0, 0.50 + 1.8 * qsurf.get("coherence", 0.0) - 1.1 * qsurf.get("spread", 0.0) - 0.55 * min(1.0, qsurf.get("phase_drift", 0.0))))
    transition_pressure = max(0.0, min(1.0, 0.25 + 1.4 * min(1.0, qsurf.get("phase_drift", 0.0)) + 0.5 * qsurf.get("edge_pull", 0.0)))
    abstraction_budget = max(0.0, min(1.0, 0.16 + 2.2 * qsurf.get("spread", 0.0) + (0.12 if dominant == "violet" else 0.0)))
    compression_pressure = max(0.0, min(1.0, 0.18 + 1.9 * qsurf.get("spread", 0.0) + 0.7 * (1.0 - harmony_score)))
    bridge_bias = max(0.0, min(1.0, 0.22 + 0.95 * min(1.0, qsurf.get("phase_drift", 0.0)) + (0.18 if "teal" in support_order else 0.0)))
    return {
        "rgb": rgb,
        "hex": rgb_hex(rgb),
        "dominant": dominant,
        "support_order": support_order,
        "directives": [lane_map.get(name, lane_map["teal"])["directive"] for name in support_order],
        "transition_modes": [lane_map.get(name, lane_map["teal"])["transition_mode"] for name in support_order],
        "repair_modes": [lane_map.get(name, lane_map["teal"])["repair_mode"] for name in support_order],
        "coherence_push": max(0.0, min(1.0, 0.40 + 2.5 * qsurf.get("coherence", 0.0) - 1.4 * qsurf.get("spread", 0.0))),
        "phase_balance": max(0.0, min(1.0, 1.0 - min(1.0, qsurf.get("phase_drift", 0.0)))),
        "harmony_score": harmony_score,
        "transition_pressure": transition_pressure,
        "abstraction_budget": abstraction_budget,
        "compression_pressure": compression_pressure,
        "bridge_bias": bridge_bias,
    }

def build_colorized_thought_surface(topic: str, stage: str, draft_so_far: str, qvec, mode: str, packet: Optional[Dict[str, Any]] = None, memory_hits: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    packet = packet or {}
    memory_hits = memory_hits or []
    palette = quantum_palette_surface(qvec)
    support_text = "\n".join([
        topic,
        stage,
        draft_so_far[-2400:],
        str(packet.get("summary", "")),
        "\n".join(str(x.get("summary", "")) for x in memory_hits[:5]),
    ]).strip()
    compression = build_semantic_compression_surface(support_text)
    anchors = compression.get("keywords", [])[:8]
    dominant = palette.get("dominant", "teal")
    stage_mission = {
        "title": "compress tension into a sharply framed payoff",
        "outline": "sequence the argument so every move earns the next",
        "section": "drive the current claim forward without losing continuity",
        "polish": "eliminate drift and make the whole article feel inevitable",
    }.get(stage, "strengthen coherence while preserving specificity")
    stage_priorities = {
        "title": ["promise", "tension", "contrast"],
        "outline": ["sequence", "handoff", "dependency"],
        "section": ["continuity", "mechanism", "payoff"],
        "polish": ["seamlessness", "compression", "cadence"],
    }.get(stage, ["continuity", "specificity", "payoff"])
    thought_cards = []
    for idx, color_name in enumerate(palette.get("support_order", [])[:4]):
        anchor = anchors[idx % len(anchors)] if anchors else topic.lower()
        directive = palette.get("directives", ["tighten coherence"])[idx]
        transition_mode = palette.get("transition_modes", ["improve continuity"])[idx]
        repair_mode = palette.get("repair_modes", ["tighten coherence"])[idx]
        thought_cards.append({
            "color": color_name,
            "anchor": anchor,
            "directive": directive,
            "transition_mode": transition_mode,
            "repair_mode": repair_mode,
            "priority": stage_priorities[idx % len(stage_priorities)],
        })
    chromatic_memory = [
        f"{hit.get('stage', 'memory')}::{(hit.get('summary') or '')[:120]}"
        for hit in memory_hits[:4]
        if hit.get('summary')
    ]
    lane_sequence = " -> ".join(card["color"] for card in thought_cards[:4]) if thought_cards else dominant
    cadence_bias = "long-short alternation" if palette.get("transition_pressure", 0.0) > 0.55 else "measured sustained cadence"
    resonance_targets = dedupe_preserve_order((anchors[:4] + [x.split('::', 1)[-1][:44] for x in chromatic_memory[:2]]))[:5]
    repair_stack = dedupe_preserve_order([card["repair_mode"] for card in thought_cards[:3]])
    cadence_plan = [
        "open with a sentence length shift" if palette.get("transition_pressure", 0.0) > 0.55 else "open with controlled momentum",
        "stabilize with medium-length mechanism sentences",
        "close with a handoff sentence that carries unresolved pressure",
    ]
    article_wave = "escalate -> stabilize -> compress -> handoff" if stage in {"section", "polish"} else "frame -> differentiate -> converge"
    synthesis_directive = (
        f"Open in {thought_cards[0]['color']} mode, stabilize with {thought_cards[1]['color']} mode, and close by preserving {thought_cards[-1]['priority']}"
        if len(thought_cards) >= 2 else f"Stay in {dominant} mode and preserve continuity"
    )
    return {
        "dominant_color": dominant,
        "hex": palette.get("hex", "#7aa7c7"),
        "mission": stage_mission,
        "coherence_push": palette.get("coherence_push", 0.5),
        "phase_balance": palette.get("phase_balance", 0.5),
        "harmony_score": palette.get("harmony_score", 0.5),
        "transition_pressure": palette.get("transition_pressure", 0.5),
        "abstraction_budget": palette.get("abstraction_budget", 0.3),
        "compression_pressure": palette.get("compression_pressure", 0.3),
        "bridge_bias": palette.get("bridge_bias", 0.4),
        "anchors": anchors,
        "semantic_summary": compression.get("summary", ""),
        "thought_cards": thought_cards,
        "lane_sequence": lane_sequence,
        "cadence_bias": cadence_bias,
        "cadence_plan": cadence_plan,
        "resonance_targets": resonance_targets,
        "repair_stack": repair_stack,
        "article_wave": article_wave,
        "chromatic_memory": chromatic_memory,
        "synthesis_directive": synthesis_directive,
        "mode_bias": "implementation precision" if mode == "software" else "chapter-scale continuity and durable conceptual depth" if mode == "longform" else "reader-facing continuity",
    }

def render_colorized_thought_surface(surface: Optional[Dict[str, Any]] = None) -> str:
    surface = surface or ACTIVE_COLOR_THOUGHT_SURFACE
    if not surface:
        return "(no colorized thought surface)"
    lines = [
        f"Dominant color: {surface.get('dominant_color', 'teal')} {surface.get('hex', '')}",
        f"Mission: {surface.get('mission', '')}",
        f"Coherence push: {float(surface.get('coherence_push', 0.0)):.3f}",
        f"Phase balance: {float(surface.get('phase_balance', 0.0)):.3f}",
        f"Harmony score: {float(surface.get('harmony_score', 0.0)):.3f}",
        f"Transition pressure: {float(surface.get('transition_pressure', 0.0)):.3f}",
        f"Abstraction budget: {float(surface.get('abstraction_budget', 0.0)):.3f}",
        f"Compression pressure: {float(surface.get('compression_pressure', 0.0)):.3f}",
        f"Bridge bias: {float(surface.get('bridge_bias', 0.0)):.3f}",
        f"Lane sequence: {surface.get('lane_sequence', '')}",
        f"Cadence bias: {surface.get('cadence_bias', '')}",
        f"Article wave: {surface.get('article_wave', '')}",
        f"Synthesis directive: {surface.get('synthesis_directive', '')}",
    ]
    if surface.get("semantic_summary"):
        lines.append("Semantic summary: " + str(surface.get("semantic_summary")))
    if surface.get("anchors"):
        lines.append("Anchors: " + ", ".join(surface.get("anchors", [])[:8]))
    if surface.get("resonance_targets"):
        lines.append("Resonance targets: " + " | ".join(surface.get("resonance_targets", [])[:5]))
    if surface.get("repair_stack"):
        lines.append("Repair stack: " + " | ".join(surface.get("repair_stack", [])[:4]))
    if surface.get("cadence_plan"):
        lines.append("Cadence plan:")
        lines.extend(f"- {x}" for x in surface.get("cadence_plan", [])[:3])
    if surface.get("chromatic_memory"):
        lines.append("Chromatic memory:")
        lines.extend(f"- {x}" for x in surface.get("chromatic_memory", [])[:3])
    for card in surface.get("thought_cards", [])[:4]:
        lines.append(f"- {card.get('color')}: anchor={card.get('anchor')} | priority={card.get('priority')} | directive={card.get('directive')} | transition={card.get('transition_mode')} | repair={card.get('repair_mode')}")
    return "\n".join(lines)

# =========================
# In-house vector memory
# =========================
MEMORY_DB_PATH = os.path.join(os.getcwd(), "writer_memory.db")
MEMORY_MAX_ROWS = 2400
MEMORY_ROTATE_KEEP = 1600
MEMORY_BANK: List[str] = []

def ensure_memory_db():
    conn = sqlite3.connect(MEMORY_DB_PATH)
    try:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS memory_surface (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            created_at TEXT NOT NULL,
            stage TEXT,
            topic TEXT,
            mode TEXT,
            title TEXT,
            text TEXT NOT NULL,
            summary TEXT,
            embedding TEXT NOT NULL,
            salience REAL DEFAULT 0.5,
            source_hash TEXT UNIQUE
        )
        CREATE TABLE IF NOT EXISTS knowledge_graph (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            created_at TEXT NOT NULL,
            source_topic TEXT,
            entity_a TEXT NOT NULL,
            relation TEXT NOT NULL,
            entity_b TEXT NOT NULL,
            weight REAL DEFAULT 0.5,
            edge_hash TEXT UNIQUE
        )
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS prompt_experiments (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            created_at TEXT NOT NULL,
            stage TEXT,
            mode TEXT,
            topic TEXT,
            prompt_hash TEXT,
            score REAL,
            notes TEXT
        )
        """)
        conn.commit()
    finally:
        conn.close()

def rotate_memory_surface():
    ensure_memory_db()
    conn = sqlite3.connect(MEMORY_DB_PATH)
    try:
        row_count = conn.execute("SELECT COUNT(*) FROM memory_surface").fetchone()[0]
        if row_count <= MEMORY_MAX_ROWS:
            return
        conn.execute("""
        DELETE FROM memory_surface
        WHERE id NOT IN (
            SELECT id FROM memory_surface
            ORDER BY salience DESC, id DESC
            LIMIT ?
        )
        """, (MEMORY_ROTATE_KEEP,))
        conn.commit()
    finally:
        conn.close()

def store_memory_surface(text: str, stage: str = "generic", topic: str = "", mode: str = "", title: str = "", salience: float = 0.5):
    clean = (text or "").strip()
    if not clean:
        return
    ensure_memory_db()
    summary = short_text_summary(clean)
    embedding = embed_text_surface(clean)
    payload = (now_ts(), stage, topic, mode, title, clean[:8000], summary, json.dumps(embedding), float(max(0.0, min(1.5, salience))), sha256_text(clean[:4000]))
    conn = sqlite3.connect(MEMORY_DB_PATH)
    try:
        conn.execute(
            "INSERT OR REPLACE INTO memory_surface (created_at, stage, topic, mode, title, text, summary, embedding, salience, source_hash) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
            payload,
        )
        conn.commit()
    finally:
        conn.close()
    rotate_memory_surface()

def retrieve_memory_surface(query: str, topic: str = "", mode: str = "", limit: int = 5) -> List[Dict[str, Any]]:
    ensure_memory_db()
    qvec = embed_text_surface((query or "") + " " + (topic or "") + " " + (mode or ""))
    conn = sqlite3.connect(MEMORY_DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        rows = conn.execute("SELECT * FROM memory_surface ORDER BY id DESC LIMIT 350").fetchall()
    finally:
        conn.close()
    scored = []
    for row in rows:
        try:
            emb = json.loads(row["embedding"])
        except Exception:
            emb = []
        lexical = 0.0
        if topic and row["topic"] and topic.lower() in row["topic"].lower():
            lexical += 0.15
        if mode and row["mode"] == mode:
            lexical += 0.08
        score = cosine_similarity(qvec, emb) + lexical + 0.05 * float(row["salience"] or 0.0)
        if score > 0.08:
            scored.append({"score": score, "summary": row["summary"], "text": row["text"], "stage": row["stage"], "topic": row["topic"], "mode": row["mode"], "title": row["title"]})
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:limit]

def render_memory_hits(hits: List[Dict[str, Any]]) -> str:
    if not hits:
        return "(no strong memory hits)"
    lines = []
    for hit in hits:
        title = hit.get("title") or hit.get("topic") or "memory"
        lines.append(f"- [{hit.get('stage', 'memory')}] {title}: {hit.get('summary', '')}")
    return "\n".join(lines[:6])

def bank_add(text: str, stage: str = "generic", topic: str = "", mode: str = "", title: str = "", salience: float = 0.5):
    clean = (text or "").strip()
    if clean:
        MEMORY_BANK.append(clean[:4000])
        store_memory_surface(clean, stage=stage, topic=topic, mode=mode, title=title, salience=salience)

def extract_entities_lightweight(text: str) -> List[str]:
    candidates = re.findall(r"\b[A-Z][A-Za-z0-9\-]{2,}(?:\s+[A-Z][A-Za-z0-9\-]{2,})*\b", text or "")
    cleaned = []
    stop = {"The", "This", "That", "These", "Those", "Section", "Conclusion"}
    for item in candidates:
        item = item.strip()
        if item and item not in stop and len(item) < 60:
            cleaned.append(item)
    return dedupe_preserve_order(cleaned)[:24]

def update_knowledge_graph(topic: str, text: str):
    entities = extract_entities_lightweight(text)
    if len(entities) < 2:
        return
    conn = sqlite3.connect(MEMORY_DB_PATH)
    try:
        for i in range(min(len(entities) - 1, 10)):
            a = entities[i]
            b = entities[i + 1]
            relation = "co_occurs_with"
            edge_hash = sha256_text(f"{topic}|{a}|{relation}|{b}")
            conn.execute(
                "INSERT OR REPLACE INTO knowledge_graph (created_at, source_topic, entity_a, relation, entity_b, weight, edge_hash) VALUES (?, ?, ?, ?, ?, ?, ?)",
                (now_ts(), topic, a, relation, b, 0.6, edge_hash),
            )
        conn.commit()
    finally:
        conn.close()

def score_article_surface(article: str) -> float:
    words = re.findall(r"\w[\w\-']*", article or "")
    length_score = min(1.0, len(words) / 1400.0)
    heading_score = min(1.0, len(re.findall(r"(?m)^##\s+", article or "")) / 6.0)
    repetition_penalty = min(0.35, max(0.0, (article.lower().count("the") / max(1, len(words))) - 0.045))
    coherence_score = min(1.0, 0.25 + heading_score * 0.35 + length_score * 0.30 + (0.20 if "## Conclusion" in (article or "") else 0.0))
    return max(0.0, min(1.0, coherence_score - repetition_penalty))

def record_prompt_experiment(stage: str, mode: str, topic: str, prompt_text: str, score: float, notes: str = ""):
    conn = sqlite3.connect(MEMORY_DB_PATH)
    try:
        conn.execute(
            "INSERT INTO prompt_experiments (created_at, stage, mode, topic, prompt_hash, score, notes) VALUES (?, ?, ?, ?, ?, ?, ?)",
            (now_ts(), stage, mode, topic, sha256_text(prompt_text[:4000]), float(score), notes[:1200]),
        )
        conn.commit()
    finally:
        conn.close()

def semantic_topic_expansion(seed_topics: List[str], mode: str, top_k: int = 8, threshold: float = 0.20) -> List[str]:
    seeds = dedupe_preserve_order([s for s in seed_topics if s])
    universe = dedupe_preserve_order(seeds + list(DEFAULT_TOPIC_BANK.get(mode, [])))
    scored = []
    for base in seeds:
        base_vec = embed_text_surface(base)
        for candidate in universe:
            if candidate == base:
                continue
            sim = cosine_similarity(base_vec, embed_text_surface(candidate))
            if sim > threshold:
                scored.append((sim, candidate))
    scored.sort(key=lambda x: x[0], reverse=True)
    out = []
    for _, cand in scored:
        if cand not in seeds and cand not in out:
            out.append(cand)
    return out[:top_k]

def gather_local_research_packet(topic: str, mode: str, dynamic_ctx: Optional[Dict[str, Any]] = None, limit: int = 6) -> Dict[str, Any]:
    queries = [topic]
    if dynamic_ctx:
        queries.extend(dynamic_ctx.get("retrieval_queries", []))
    hits = []
    for q in queries[:4]:
        hits.extend(retrieve_memory_surface(q, topic=topic, mode=mode, limit=limit))
    uniq = []
    seen = set()
    for hit in hits:
        key = hit.get("summary", "")
        if key and key not in seen:
            uniq.append(hit)
            seen.add(key)
    summary_lines = [f"- {h.get('summary', '')}" for h in uniq[:limit] if h.get('summary')]
    return {"hits": uniq[:limit], "summary": "\n".join(summary_lines) if summary_lines else "(no strong local research yet)"}

CONTROL_TAG_SPECS = {
    "action": "describe the next structural move or continuity handoff",
    "focus": "name the exact concept, subsystem, or argument center this chunk should stay anchored on",
    "bridge": "state the connection from the previous paragraph or section to the next move",
    "evidence": "state what kind of support is needed next: mechanism, example, edge case, or consequence",
    "tempo": "set the pacing intent: accelerate, dwell, compress, or pivot",
    "avoid": "name a phrase pattern, repeated idea, or weak move to avoid in this chunk",
    "color": "assign a hidden color lane to the current thought path for coherence control",
    "thought": "state the hidden reasoning lane or semantic anchor driving this chunk",
    "repair": "state the local weakness this chunk should quietly correct",
    "resonance": "state what prior anchor, claim, or section should echo here",
    "cadence": "state the sentence rhythm target for this chunk",
    "handoff": "state exactly what unresolved pressure should be handed to the next unit",
}
CONTROL_TAG_PATTERNS = {name: re.compile(rf"\[{name}\](.*?)\[/{name}\]", re.S | re.I) for name in CONTROL_TAG_SPECS}

def extract_control_tags(text: str) -> Dict[str, List[str]]:
    found = {name: [] for name in CONTROL_TAG_SPECS}
    if not text:
        return found
    for name, pattern in CONTROL_TAG_PATTERNS.items():
        found[name] = [m.strip() for m in pattern.findall(text) if m.strip()]
    return found

def strip_control_tags(text: str) -> str:
    if not text:
        return ""
    cleaned = text
    for pattern in CONTROL_TAG_PATTERNS.values():
        cleaned = pattern.sub("", cleaned)
    return cleaned.strip()

def render_control_tag_memory(tag_map: Dict[str, List[str]], keep_last: int = 2) -> str:
    lines = []
    for name in ["action", "focus", "bridge", "evidence", "tempo", "avoid", "color", "thought", "repair", "resonance", "cadence", "handoff"]:
        vals = tag_map.get(name, [])[-keep_last:]
        if vals:
            for val in vals:
                lines.append(f"- {name}: {val}")
    return "\n".join(lines) if lines else "(none yet)"

def action_protocol(stage: str, mode: str) -> str:
    return f"""
You may optionally use hidden steering tags before the visible prose.
Allowed tags:
[action]next structural move[/action]
[focus]main concept to stay centered on[/focus]
[bridge]how this chunk connects to what came before or what comes next[/bridge]
[evidence]what kind of support should appear next[/evidence]
[tempo]accelerate, dwell, compress, or pivot[/tempo]
[avoid]what repetition, cliche, or weak move to avoid[/avoid]
[color]hidden chromatic thought lane such as teal, amber, jade, cobalt, or crimson[/color]
[thought]hidden semantic anchor or reasoning lane for the chunk[/thought]
[repair]hidden weakness to quietly fix in this chunk[/repair]
[resonance]what earlier anchor or section should echo here[/resonance]
[cadence]sentence rhythm target such as clipped, measured, or elastic[/cadence]
[handoff]what unresolved pressure should be passed forward[/handoff]

Tag rules:
- All tags are optional.
- Use at most 2 tags per response chunk.
- Keep each tag under 18 words.
- Tags are for hidden coherence control only.
- After the tags, write the actual {stage} output immediately.
- Never mention that the tags exist.
- In {mode} mode, prefer tags that sharpen continuity, scope, causality, specificity, and handoff quality.
""".strip()

def build_coherence_contract(stage: str, mode: str) -> str:
    if stage == "title":
        return "Every title must imply a distinct angle, concrete payoff, real tension, and a coherent semantic center. Use the hidden color lane only to sharpen the promise, never to become visible style clutter. Preserve one dominant semantic anchor and avoid title candidates that merely permute the same phrasing."
    if stage == "outline":
        return "The outline must progress cumulatively. Each section should unlock something the next section depends on, and the section chain should follow one stable semantic compression spine rather than scattered subtopics. The chromatic progression should have direction: opener lane, bridge lane, stabilization lane, payoff lane."
    if stage == "section":
        return "Within the section, every paragraph should either deepen, test, exemplify, or transition the core claim. Keep one dominant thought lane, preserve anchor keywords across paragraph turns, use hidden chromatic control only to strengthen coherence, and let cadence, resonance, and handoff stay legible beneath the prose."
    if stage == "polish":
        return "The full piece must read like a single authored argument with no obvious seam lines between generation passes. Resolve semantic drift, align section colors into one article-wide progression, preserve the strongest retrieval anchors, and make repair moves feel absorbed rather than pasted on."
    return "Maintain continuity, specificity, structural intent, semantic anchor stability, a coherent hidden thought lane, and explicit resonance between distant parts of the piece."

# =========================
# Generation wrappers
# =========================
def llm_call(prompt: str, max_tokens: int = 350, temperature: float = 0.85, top_p: float = 0.92, repeat_penalty: float = 1.08, stop: Optional[List[str]] = None) -> str:
    out = LLM(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        repeat_penalty=repeat_penalty,
        stop=stop or [],
    )
    return out["choices"][0]["text"]

def generate_chunked(prompt: str, max_total_new: int = 700, chunk_new_init: int = 220, srdi_enable: bool = True, human_style: bool = True, memory_query: str = "", surface_hint: Optional[Dict[str, Any]] = None, temperature_override: Optional[float] = None, repeat_penalty_override: Optional[float] = None, top_p_override: Optional[float] = None, stage_name: str = "generic", mode_name: str = "blog") -> str:
    generated = ""
    remaining = max_total_new
    chunk = chunk_new_init
    loops = 0
    qvec_hint = surface_hint.get("qvec") if isinstance(surface_hint, dict) else None
    hits = retrieve_memory_surface_quantum_color(memory_query or prompt[:500], topic=str(surface_hint.get("topic", "")) if isinstance(surface_hint, dict) else "", mode=mode_name, qvec=qvec_hint, limit=4) if qvec_hint is not None else retrieve_memory_surface(memory_query or prompt[:500], limit=4)
    memory_surface_text = render_memory_hits(hits)
    semantic_memory = build_semantic_compression_surface("\n".join((hit.get("summary") or "") for hit in hits[:4]))
    color_surface = build_colorized_thought_surface(memory_query or stage_name, stage_name, prompt, qvec_hint if qvec_hint is not None else quantum_state(0.5, 0.5, 0.5, loops + 1), mode_name, packet=ACTIVE_RESEARCH_PACKET, memory_hits=hits)
    surface_hint = surface_hint or {}
    seen_hashes = set()
    while remaining > 0 and loops < 10:
        loops += 1
        temp = temperature_override if temperature_override is not None else max(0.55, min(1.15, (0.90 if human_style else 0.80) + 0.12 * float(surface_hint.get("entropic_gain", 0.0)) - 0.05 * loops))
        top_p = top_p_override if top_p_override is not None else max(0.82, min(0.98, 0.91 + 0.05 * float(surface_hint.get("coherence", 0.0))))
        repeat_penalty = repeat_penalty_override if repeat_penalty_override is not None else max(1.02, min(1.22, 1.07 + 0.07 * float(surface_hint.get("phase_drift", 0.0))))
        continuation = "Continue the same output without restarting. Expand only where needed, preserve continuity, and do not repeat prior phrases." if generated else "Start strongly and obey the prompt precisely."
        prior_tags = extract_control_tags(generated)
        tag_memory = render_control_tag_memory(prior_tags, keep_last=2)
        current_prompt = f"{prompt}\n\nCoherence contract:\n{build_coherence_contract(stage_name, mode_name)}\n\nAction protocol:\n{action_protocol(stage_name, mode_name)}\n\nMemory surface:\n{memory_surface_text}\n\nSemantic compression surface:\n{render_semantic_compression_surface(semantic_memory)}\n\nQuantum colorized thought surface:\n{render_colorized_thought_surface(color_surface)}\n\nPrior control tags:\n{tag_memory}\n\nCurrent draft:\n{strip_control_tags(generated)[-2200:] if generated else '(empty)'}\n\nInstruction: {continuation}"
        part = llm_call(
            current_prompt,
            max_tokens=min(chunk, remaining),
            temperature=temp,
            top_p=top_p,
            repeat_penalty=repeat_penalty,
            stop=["\n\n[END]", "\n\n### INTERNAL", "\n\n```output_end"],
        )
        clean = part.strip()
        if not clean:
            break
        stamp = sha256_text(clean[:800])
        if stamp in seen_hashes:
            break
        seen_hashes.add(stamp)
        generated += ("\n\n" if generated and not generated.endswith("\n") else "") + clean
        remaining -= max(1, len(strip_control_tags(clean).split()))
        if srdi_enable:
            pressure = max(0.55, min(1.55, float(surface_hint.get("chunk_pressure", 0.92))))
            if loops % 2 == 0:
                chunk = max(100, int(chunk * (0.80 + 0.08 * pressure)))
            else:
                chunk = min(max_total_new, max(110, int(chunk * (0.92 + 0.06 * pressure))))
    tags = extract_control_tags(generated)
    for tag_name, values in tags.items():
        if values:
            store_memory_surface("\n".join(values), stage=f"tag_{tag_name}_{stage_name}", topic=memory_query, mode=mode_name, title="", salience=0.56 if tag_name == "avoid" else 0.60)
    return strip_control_tags(generated)

# =========================
# Logging
# =========================
RUN_LOG: List[Dict[str, Any]] = []

def log_event(item: Dict[str, Any]):
    item = dict(item)
    item["timestamp"] = now_ts()
    RUN_LOG.append(item)
    if item.get("consensus"):
        store_memory_surface(str(item.get("consensus")), stage=str(item.get("stage", "log")), topic=str(item.get("topic", "")), mode=str(item.get("mode", "")), title=str(item.get("title", "")), salience=0.62)

ensure_memory_db()

# =========================
# Styles / settings / agents
# =========================
@dataclass
class HumanStyleProfile:
    sentence_variety: float = 0.8
    metaphor_rate: float = 0.2
    humor_rate: float = 0.08
    directness: float = 0.85
    narrative_pull: float = 0.45
    technical_density: float = 0.7

@dataclass
class BlogSettings:
    topic: str
    mode: str = "blog"
    root_prompt: str = ""
    audience: str = "general readers"
    tone: str = "clear, practical, lightly witty"
    perspective: str = "second-person guidance"
    reading_grade: str = "8-10"
    target_words: int = 1600
    section_count: int = 6
    human_style_mode: bool = True
    title_hint: Optional[str] = None
    keywords: Optional[str] = None
    author_name: Optional[str] = None
    include_front_matter: bool = True
    include_meta_description: bool = True
    forum_personas: int = 0
    forum_rounds: int = 0
    ghostwriter_passes: int = 0
    creativity_bias: float = 0.95
    use_forum_on_sections: str = ""

@dataclass
class Agent:
    name: str
    role: str
    brief: str
    memory: List[str] = field(default_factory=list)

    def add(self, item: str):
        if item and item.strip():
            self.memory.append(item.strip()[:2000])

def make_agents() -> List[Agent]:
    return [
        Agent("Strategist", "Structure designer", "Find the strongest angle, pacing, progression, and section dependency chain."),
        Agent("Researcher", "Specificity hunter", "Add grounded detail, realistic examples, implementation nuance, and non-obvious distinctions."),
        Agent("Stylist", "Voice shaper", "Make the prose human, vivid, rhythmically varied, and difficult to mistake for template writing."),
        Agent("Systems", "Constraint architect", "Track invariants, edge cases, interfaces, and operational consequences."),
        Agent("Chromatic", "Colorized thought architect", "Use the quantum color surface, semantic compression, and memory resonance to choose the best thought lane for coherence, continuity, and argument pressure."),
        Agent("Editor", "Clarity enforcer", "Cut fluff, resolve ambiguity, remove repetition, and sharpen transitions."),
    ]

# =========================
# Prompt builders
# =========================
SYSTEM_INVENTIONS = [
    "Counterfactual Lattice: every section must pressure-test itself against a plausible alternative or objection path before landing.",
    "Entropy Shear Routing: chunk size, pacing, and exploration depth bend according to the live entropic surface instead of staying static.",
    "Vector Echo Distillation: retrieved memory is compressed into a short guidance surface before each generation step to reduce drift without overstuffing context.",
    "Rotating Memory Bloom: salience-weighted SQLite memory keeps high-signal fragments alive while pruning low-yield fragments automatically.",
    "Chromatic Thought Lanes: a quantum-derived color spectrum assigns hidden reasoning lanes that regulate tension, bridge density, and explanatory pressure.",
    "Semantic Compression Weave: summa and NLTK compress retrieved material into anchor sentences and keyword rails before prompting.",
]
SYSTEM_IDEA_REGISTRY = [
    "Idea 1 - Multi-Agent Writing Architecture",
    "Idea 2 - Retrieval Augmented Writing",
    "Idea 3 - Self-Critique Loop",
    "Idea 4 - Semantic Topic Expansion",
    "Idea 5 - Long-Form Memory System",
    "Idea 6 - Knowledge Graph Integration",
    "Idea 7 - Adaptive Prompt Optimization",
    "Idea 8 - Hybrid Quantum-AI Content Engine",
    "Idea 9 - Automatic Research Gathering",
    "Idea 10 - Autonomous Blog Factory",
]
ACTIVE_SYSTEM_UPGRADES: List[Dict[str, str]] = []
ACTIVE_RESEARCH_PACKET: Dict[str, Any] = {"summary": "(none yet)", "hits": []}
ACTIVE_COLOR_THOUGHT_SURFACE: Dict[str, Any] = {"mission": "(none yet)", "anchors": []}

def default_refined_system_upgrades() -> List[Dict[str, str]]:
    return [
        {"name": "Fractal Compression Ladders", "principle": "compress each completed section into a reusable mini-brief that can guide later drafting", "prompt_directive": "periodically compress the strongest logic into a short reusable brief", "chunk_directive": "end the section by distilling its governing rule", "memory_directive": "store section summaries as reusable compression notes"},
        {"name": "Causal Backfill Pulses", "principle": "detect implied assumptions and repair missing prerequisite explanation only where needed", "prompt_directive": "if a claim depends on unstated setup, inject a precise repair explanation", "chunk_directive": "patch missing causal prerequisites before advancing", "memory_directive": "mark missing-prerequisite repairs as high-salience memory"},
        {"name": "Memory Resonance Gating", "principle": "admit retrieved memory only when similarity and salience both justify the context cost", "prompt_directive": "use retrieved memory only when it sharpens specificity or continuity", "chunk_directive": "blend memory only when it adds net value", "memory_directive": "prefer fewer higher-signal recalls over broad memory stuffing"},
        {"name": "Phase-Locked Revision Windows", "principle": "revise high-drift spans more aggressively than already-stable spans", "prompt_directive": "rewrite unstable spans harder than stable ones", "chunk_directive": "polish rough transitions instead of flattening the whole section", "memory_directive": "track unstable spans for later targeted revision"},
        {"name": "Argument Load Balancing", "principle": "redistribute explanation density so no section carries too much abstract load", "prompt_directive": "spread difficult explanation across section boundaries instead of dumping it in one place", "chunk_directive": "offload overflow into example, objection, or next-step chunks", "memory_directive": "remember overloaded sections as balancing targets"},
        {"name": "Handoff Tension Anchors", "principle": "end each section on the unresolved pressure that the next section resolves", "prompt_directive": "close sections on an open tension that naturally leads onward", "chunk_directive": "write endings as controlled handoffs, not full stops", "memory_directive": "store handoff sentences as transition anchors"},
        {"name": "Evidence Shape Cycling", "principle": "rotate support styles between mechanism, example, contrast, and consequence to prevent monotony", "prompt_directive": "vary proof style between adjacent sections", "chunk_directive": "switch evidence shape if the previous chunk used the same one", "memory_directive": "tag evidence style to avoid repetition"},
        {"name": "Drift-Triggered Section Audits", "principle": "when stylistic or conceptual drift rises, insert a tiny audit before continuing", "prompt_directive": "pause briefly to realign when the draft drifts from the outline", "chunk_directive": "audit and realign before continuing after drift", "memory_directive": "store drift audits as corrective memory"},
    ]

def normalize_refined_ideas(items: Any) -> List[Dict[str, str]]:
    out: List[Dict[str, str]] = []
    if not isinstance(items, list):
        return out
    for item in items:
        if not isinstance(item, dict):
            continue
        name = str(item.get("name", "")).strip()
        principle = str(item.get("principle", "")).strip()
        prompt_directive = str(item.get("prompt_directive", "")).strip()
        chunk_directive = str(item.get("chunk_directive", "")).strip()
        memory_directive = str(item.get("memory_directive", "")).strip()
        if name and principle and prompt_directive:
            out.append({
                "name": name,
                "principle": principle,
                "prompt_directive": prompt_directive,
                "chunk_directive": chunk_directive or prompt_directive,
                "memory_directive": memory_directive or principle,
            })
    return out

def render_active_upgrades(upgrades: Optional[List[Dict[str, str]]] = None, field: str = "principle") -> str:
    upgrades = upgrades if upgrades is not None else ACTIVE_SYSTEM_UPGRADES
    if not upgrades:
        return "(none active)"
    lines = []
    for up in upgrades[:6]:
        lines.append(f"- {up.get('name', 'Upgrade')}: {up.get(field, '')}")
    return "\n".join(lines)

def render_research_packet(packet: Optional[Dict[str, Any]] = None) -> str:
    packet = packet if packet is not None else ACTIVE_RESEARCH_PACKET
    if not isinstance(packet, dict):
        return "(none yet)"
    lines = [str(packet.get("summary", "(none yet)"))]
    compression = packet.get("compression") or {}
    if compression:
        lines.append(render_semantic_compression_surface(compression))
    color_surface = packet.get("color_surface") or {}
    if color_surface:
        lines.append(render_colorized_thought_surface(color_surface))
    return "\n\n".join(x.strip() for x in lines if str(x).strip()) or "(none yet)"

ADVANCED_IDEA_STACK = [
    {
        "idea": "Idea 1 - Multi-Agent Writing Architecture",
        "advanced": "Hierarchical specialist swarm with strategist, researcher, systems, stylist, critic, verifier, and handoff-controller roles.",
        "system_hook": "Use role-specialized prompts, cross-agent memory, and stage-specific synthesis contracts.",
    },
    {
        "idea": "Idea 2 - Retrieval Augmented Writing",
        "advanced": "Hybrid retrieval that mixes vector similarity, lexical overlap, graph neighbors, salience, and stage-sensitive reranking.",
        "system_hook": "Use vector recall plus graph context and memory gating before each stage and chunk.",
    },
    {
        "idea": "Idea 3 - Self-Critique Loop",
        "advanced": "Layered critique passes separating factual weakness, structural drift, style drag, and missing support into different repair modes.",
        "system_hook": "Run targeted critique cycles instead of one generic rewrite loop.",
    },
    {
        "idea": "Idea 4 - Semantic Topic Expansion",
        "advanced": "Topic neighborhoods are expanded through vector similarity, memory resonance, and graph-adjacent entities rather than a static bank.",
        "system_hook": "Expand seeds using vectors, memory, and extracted entity trails.",
    },
    {
        "idea": "Idea 5 - Long-Form Memory System",
        "advanced": "Long-form memory is split into draft memory, compression memory, corrective memory, and high-salience evergreen memory.",
        "system_hook": "Persist articles, critiques, tags, section summaries, and repairs with salience-aware rotation.",
    },
    {
        "idea": "Idea 6 - Knowledge Graph Integration",
        "advanced": "Entity extraction feeds a lightweight evolving graph used for adjacency retrieval, topic discovery, and continuity reinforcement.",
        "system_hook": "Update a graph after each run and query neighbors during retrieval and planning.",
    },
    {
        "idea": "Idea 7 - Adaptive Prompt Optimization",
        "advanced": "Prompt strategies become scored policies with reusable winners chosen per stage, mode, and article quality signature.",
        "system_hook": "Track prompt experiments and select prompt-policy overlays before generation.",
    },
    {
        "idea": "Idea 8 - Hybrid Quantum-AI Content Engine",
        "advanced": "Quantum surfaces do more than styling: they modulate exploration, chunk pressure, revision aggression, and section pacing.",
        "system_hook": "Use coherence, spread, phase drift, and edge pull to steer chunking and repair behavior.",
    },
    {
        "idea": "Idea 9 - Automatic Research Gathering",
        "advanced": "Research becomes a synthesized local packet made of retrieved memory, graph context, compression notes, and relevant prior critiques.",
        "system_hook": "Inject a staged research brief before title, outline, sections, and polish.",
    },
    {
        "idea": "Idea 10 - Autonomous Blog Factory",
        "advanced": "A scheduler plans idea generation, research, writing, revision, scoring, memory updates, and next-wave topic selection in batches.",
        "system_hook": "Use a lightweight factory planner and batch runner over the existing notebook engine.",
    },
    {
        "idea": "Idea 11 - Colorized Thought Agent",
        "advanced": "A specialized chromatic reasoning agent maps quantum color lanes to hidden thought modes that regulate tension, transition density, abstraction depth, and coherence repair.",
        "system_hook": "Derive a colorized thought surface from PennyLane state vectors, feed it into retrieval and prompting, and allow hidden [color] and [thought] tags to stabilize generation.",
    },
    {
        "idea": "Idea 12 - Semantic Compression Weave",
        "advanced": "Summa and NLTK compress retrieved material into keyword rails, anchor sentences, and summary spines so prompts can stay dense without bloating context.",
        "system_hook": "Build semantic compression surfaces for memory hits, research packets, agent prompts, synthesis prompts, and chunk-level planning.",
    },
]

def render_advanced_idea_stack() -> str:
    return "\n".join(f"- {x['idea']}: {x['advanced']}" for x in ADVANCED_IDEA_STACK)

def render_advanced_system_hooks() -> str:
    return "\n".join(f"- {x['idea']}: {x['system_hook']}" for x in ADVANCED_IDEA_STACK)

@dataclass
class PromptPolicy:
    name: str
    directive: str
    when_to_use: str

@dataclass
class ResearchArtifact:
    query: str
    summary: str
    support_type: str
    score: float

def prompt_policy_library(mode: str) -> List[PromptPolicy]:
    shared = [
        PromptPolicy("Specificity First", "Prefer mechanism, consequence, and example before abstraction.", "Use when the draft risks becoming generic."),
        PromptPolicy("Continuity Spine", "Carry forward the unresolved tension from the prior section before adding new claims.", "Use when long-form coherence matters most."),
        PromptPolicy("Evidence Cycling", "Rotate support style between mechanism, example, contrast, and implication.", "Use when adjacent paragraphs feel repetitive."),
        PromptPolicy("Chromatic Thought Rail", "Use the dominant quantum color lane plus semantic anchors to regulate abstraction, pressure, and handoff quality.", "Use when coherence drift or conceptual scatter appears."),
    ]
    if mode == "software":
        shared.extend([
            PromptPolicy("Contract First", "State interfaces, invariants, and ownership before operational detail.", "Use in architecture-heavy output."),
            PromptPolicy("Failure Surface", "Always pair a design path with failure modes and mitigations.", "Use when reliability matters."),
        ])
    elif mode == "longform":
        shared.extend([
            PromptPolicy("Chapter Gravity", "Write with chapter-level momentum, letting ideas unfold with durable depth instead of article-speed compression.", "Use in manuscript-scale writing."),
            PromptPolicy("Framework Recurrence", "Introduce reusable frameworks and let them recur across later chapters with variation.", "Use when writing book-length conceptual work."),
        ])
    else:
        shared.extend([
            PromptPolicy("Narrative Torque", "Open with pressure, then cash it out with concrete explanation.", "Use in blog-style sections."),
            PromptPolicy("Reader Transfer", "End sections with a portable heuristic or changed mental model.", "Use when sections need stronger payoff."),
        ])
    return shared

def choose_prompt_policy(topic: str, mode: str, stage: str) -> PromptPolicy:
    policies = prompt_policy_library(mode)
    prior = retrieve_memory_surface(f"prompt_experiments {topic} {stage} {mode}", topic=topic, mode=mode, limit=3)
    if prior and stage in {"section", "polish"}:
        for policy in policies:
            if "Continuity" in policy.name or "Contract" in policy.name:
                return policy
    if stage == "outline":
        return next((p for p in policies if "Continuity" in p.name), policies[0])
    if stage == "section":
        return next((p for p in policies if ("Contract" in p.name if mode == "software" else "Chapter" in p.name if mode == "longform" else "Narrative" in p.name)), policies[0])
    return policies[0]

def graph_neighbors(topic: str, limit: int = 10) -> List[str]:
    ensure_memory_db()
    conn = sqlite3.connect(MEMORY_DB_PATH)
    try:
        rows = conn.execute(
            "SELECT entity_a, entity_b, source_topic FROM knowledge_graph WHERE source_topic = ? OR entity_a LIKE ? OR entity_b LIKE ? ORDER BY weight DESC, id DESC LIMIT ?",
            (topic, f"%{topic}%", f"%{topic}%", limit),
        ).fetchall()
    finally:
        conn.close()
    out = []
    for a, b, st in rows:
        for item in [a, b, st]:
            if item and item not in out:
                out.append(item)
    return out[:limit]

def retrieve_memory_surface_advanced(query: str, topic: str = "", mode: str = "", limit: int = 6) -> List[Dict[str, Any]]:
    hits = retrieve_memory_surface(query, topic=topic, mode=mode, limit=max(limit * 2, 10))
    neighbors = set(graph_neighbors(topic, limit=8)) if topic else set()
    reranked = []
    for hit in hits:
        graph_bonus = 0.0
        txt = (hit.get("text") or "") + " " + (hit.get("summary") or "")
        if any(n and n in txt for n in neighbors):
            graph_bonus += 0.08
        stage_bonus = 0.05 if (hit.get("stage") or "").startswith("final_article") else 0.0
        reranked.append({**hit, "score": hit.get("score", 0.0) + graph_bonus + stage_bonus})
    reranked.sort(key=lambda x: x.get("score", 0.0), reverse=True)
    return reranked[:limit]

def retrieve_memory_surface_quantum_color(query: str, topic: str = "", mode: str = "", qvec = None, limit: int = 6) -> List[Dict[str, Any]]:
    hits = retrieve_memory_surface_advanced(query, topic=topic, mode=mode, limit=max(limit * 2, 10))
    if not hits:
        return []
    query_keywords = set(combine_keyword_surfaces(summa_keyword_surface(query, words=10), nltk_keyword_surface(query, limit=10), limit=12))
    palette = quantum_palette_surface(qvec) if qvec is not None else {"coherence_push": 0.5, "phase_balance": 0.5, "dominant": "teal"}
    reranked = []
    for hit in hits:
        support = build_semantic_compression_surface((hit.get("summary") or "") + "\n" + (hit.get("text") or ""))
        hit_keywords = set(support.get("keywords", []))
        overlap = len(query_keywords & hit_keywords) / max(1, len(query_keywords) or len(hit_keywords) or 1)
        color_bonus = 0.04 if palette.get("dominant") in ("teal", "jade", "cobalt") and (hit.get("stage") or "").startswith(("section", "consensus", "final")) else 0.0
        quantum_bonus = 0.05 * float(palette.get("coherence_push", 0.5)) + 0.03 * float(palette.get("phase_balance", 0.5))
        reranked.append({**hit, "score": float(hit.get("score", 0.0)) + overlap * 0.22 + quantum_bonus + color_bonus, "semantic_support": support})
    reranked.sort(key=lambda x: x.get("score", 0.0), reverse=True)
    return reranked[:limit]

def build_research_packet_advanced(topic: str, mode: str, dynamic_ctx: Optional[Dict[str, Any]] = None, limit: int = 8, qvec = None) -> Dict[str, Any]:
    base = gather_local_research_packet(topic, mode, dynamic_ctx=dynamic_ctx, limit=limit)
    neighbors = graph_neighbors(topic, limit=8)
    advanced_hits = retrieve_memory_surface_quantum_color(topic, topic=topic, mode=mode, qvec=qvec, limit=limit)
    artifacts: List[ResearchArtifact] = []
    for hit in advanced_hits[:limit]:
        support_type = "example"
        summary = hit.get("summary", "")
        if any(word in summary.lower() for word in ["edge", "failure", "risk"]):
            support_type = "edge_case"
        elif any(word in summary.lower() for word in ["because", "causal", "mechanism"]):
            support_type = "mechanism"
        artifacts.append(ResearchArtifact(query=topic, summary=summary, support_type=support_type, score=float(hit.get("score", 0.0))))
    lines = [f"- ({a.support_type}) {a.summary}" for a in artifacts if a.summary]
    if neighbors:
        lines.append("- Graph neighbors: " + ", ".join(neighbors[:6]))
    merged_text = "\n".join([base.get("summary", "")] + [a.summary for a in artifacts if a.summary])
    compression = build_semantic_compression_surface(merged_text)
    color_surface = build_colorized_thought_surface(topic, "research", merged_text, qvec if qvec is not None else quantum_state(0.5, 0.5, 0.5, 3), mode, packet=base, memory_hits=advanced_hits)
    summary = "\n".join(lines) if lines else base.get("summary", "(no research)")
    if compression.get("summary"):
        summary = summary + "\n- Semantic compression: " + compression.get("summary", "")
    if compression.get("keywords"):
        summary = summary + "\n- Semantic keywords: " + ", ".join(compression.get("keywords", [])[:10])
    summary = summary.strip() or "(no research)"
    return {"hits": advanced_hits, "neighbors": neighbors, "artifacts": [asdict(a) for a in artifacts], "summary": summary, "compression": compression, "color_surface": color_surface}

def stage_contract(stage: str, mode: str) -> str:
    if stage == "title":
        return "Produce 8 title candidates with concrete tension, strong specificity, an obvious reader payoff, and a tight semantic center. Each title should imply a different angle, not a paraphrase. No generic titles, no vague buzzwords, no dead metaphors, no drift from the core retrieval anchors."
    if stage == "outline":
        return "Produce a section sequence with escalating value. Every section must earn its place, avoid overlap, and set up the next section through dependency or contrast. Use a semantic compression spine, preserve retrieval continuity, and return headers only."
    if stage == "section":
        return "Produce a polished markdown section with real substance, continuity, useful specificity, and stable semantic anchors. The section must open with momentum, deepen through mechanism or example, close with a line that hands off naturally to what follows, and stay on one deliberate thought lane. No filler throat-clearing."
    if stage == "polish":
        return "Unify the whole piece, tighten repetition, improve transitions, preserve the strongest concrete material, align the semantic compression surface across sections, and make the document feel authored in one deliberate pass rather than assembled from fragments."
    return "Return only content directly usable for this stage, with continuity, semantic anchor stability, and structural intent preserved."

def build_advanced_prompt_contract(stage: str, mode: str, color_surface: Optional[Dict[str, Any]] = None) -> str:
    surface = color_surface or ACTIVE_COLOR_THOUGHT_SURFACE or {}
    dominant = surface.get("dominant_color", "teal")
    lane_sequence = surface.get("lane_sequence", dominant)
    anchors = ", ".join(surface.get("anchors", [])[:6]) or "(none)"
    return f"""
Advanced prompt contract for {stage}/{mode}:
- Dominant chromatic lane: {dominant}
- Lane sequence: {lane_sequence}
- Semantic anchors to preserve: {anchors}
- Coherence push: {float(surface.get('coherence_push', 0.0)):.3f}
- Harmony score: {float(surface.get('harmony_score', 0.0)):.3f}
- Transition pressure: {float(surface.get('transition_pressure', 0.0)):.3f}
- Compression pressure: {float(surface.get('compression_pressure', 0.0)):.3f}
- Bridge bias: {float(surface.get('bridge_bias', 0.0)):.3f}
- Respect hidden resonance, cadence, repair, and handoff controls where useful.
- Prefer changes that simultaneously improve specificity, continuity, and forward pull.
""".strip()

def build_style_profile_llm(topic: str, qvec, entropy: float, creativity: float) -> HumanStyleProfile:
    rgb = quantum_rgb(qvec)
    seed = f"""
We are shaping a writing voice for this topic: "{topic}".
Quantum color signal: rgb{rgb}
Entropy: {entropy:.3f}
Creativity bias: {creativity:.2f}
Target outcome: vivid, expert, concrete writing that sounds authored rather than generated.

Interpretation goals:
- Sentence variety should create rhythm, not randomness.
- Metaphor rate should stay disciplined and only appear when it clarifies.
- Humor should be light and rare unless the topic naturally supports it.
- Directness should stay high enough to avoid hedging sludge.
- Narrative pull should create forward motion without becoming fluffy.
- Technical density should track the likely sophistication of a serious reader.

Return JSON only with keys:
sentence_variety, metaphor_rate, humor_rate, directness, narrative_pull, technical_density

Values must be numbers between 0 and 1.
""".strip()
    resp = generate_chunked(seed, max_total_new=180, chunk_new_init=100, srdi_enable=False, human_style=True, stage_name="style_profile", mode_name="meta")
    js = extract_json_object(resp)
    if not js:
        return HumanStyleProfile()
    return HumanStyleProfile(
        sentence_variety=clamp01(js.get("sentence_variety", 0.8), 0.8),
        metaphor_rate=clamp01(js.get("metaphor_rate", 0.2), 0.2),
        humor_rate=clamp01(js.get("humor_rate", 0.08), 0.08),
        directness=clamp01(js.get("directness", 0.85), 0.85),
        narrative_pull=clamp01(js.get("narrative_pull", 0.45), 0.45),
        technical_density=clamp01(js.get("technical_density", 0.7), 0.7),
    )

def build_agent_prompt(
    ag: Agent,
    all_agents: List[Agent],
    qsignal: str,
    qstate_vec,
    cpu: float,
    ram: float,
    entropy: float,
    stage: str,
    settings: BlogSettings,
    style: HumanStyleProfile,
    title: Optional[str],
    outline: List[str],
    section_index: Optional[int],
    draft_so_far: str,
    drift_changes: Optional[Dict[str, Any]] = None,
) -> str:
    outline_txt = "\n".join(f"- {x}" for x in outline) if outline else "(none yet)"
    section_target = outline[section_index] if (section_index is not None and section_index < len(outline)) else ""
    mode_instructions = (
        "This is a software-writing task. Prefer architecture, contracts, APIs, edge cases, implementation reasoning, deployment, testability, and tradeoffs."
        if settings.mode == "software"
        else "This is a longform manuscript-writing task. Prefer chapter gravity, deep explanation, recurring frameworks, durable original ideas, and continuity strong enough to support book-scale reading."
        if settings.mode == "longform"
        else "This is a blog-writing task. Prefer narrative flow, insight density, examples, readability, and strong transitions with no generic filler."
    )
    drift_txt = json.dumps(drift_changes or {}, ensure_ascii=False, indent=2)
    memory_query = f"{settings.topic} {title or ''} {section_target} {stage}"
    raw_memory_hits = retrieve_memory_surface_quantum_color(memory_query, topic=settings.topic, mode=settings.mode, qvec=qstate_vec, limit=5)
    memory_hits = render_memory_hits(raw_memory_hits)
    semantic_memory = build_semantic_compression_surface("\n".join((hit.get("summary") or "") for hit in raw_memory_hits[:5]))
    color_surface = build_colorized_thought_surface(settings.topic, stage, draft_so_far, qstate_vec, settings.mode, packet=ACTIVE_RESEARCH_PACKET, memory_hits=raw_memory_hits)
    role_protocol = {
        "Strategist": [
            "Design a progression where each move changes the reader's state, not just the wording.",
            "Use the chromatic lane sequence to decide where to escalate, stabilize, bridge, or compress.",
            "Prefer dependency chains and tension handoffs over parallel bullet-like sequencing.",
        ],
        "Researcher": [
            "Exploit semantic anchors and retrieved memory to add specific support that feels earned.",
            "When a claim is thin, add mechanism, example, edge case, or practical consequence rather than filler detail.",
            "Use chromatic repair modes to decide whether to sharpen, stabilize, warn, or compress.",
        ],
        "Stylist": [
            "Keep the prose authored and alive, but subordinate flourish to continuity and semantic pressure.",
            "Use cadence bias and abstraction budget from the color surface to control rhythm.",
            "Vary texture without breaking the current thought lane.",
        ],
        "Systems": [
            "Protect invariants, definitions, interfaces, and consequences across the whole response.",
            "If a section implies structure, make the structure explicit enough to be usable.",
            "Favor cobalt or jade style repairs when ambiguity threatens coherence.",
        ],
        "Chromatic": [
            "Interpret the PennyLane quantum surface as a hidden reasoning palette, not as visible decoration.",
            "Choose the best lane sequence for coherence: which color should open, bridge, stabilize, and close.",
            "Name what semantic anchors must recur so the article feels like one mind thinking through a problem.",
        ],
        "Editor": [
            "Detect drift, repetition, handoff weakness, and false emphasis before they propagate.",
            "Use compression pressure to decide what to cut and bridge bias to decide what to reconnect.",
            "Favor sharp local repairs over flattening the entire passage.",
        ],
    }
    operating_steps = role_protocol.get(ag.name, [
        "Maintain continuity, specificity, and structural discipline.",
        "Use the semantic and chromatic surfaces as hidden control rails.",
        "Prefer useful movement over decorative language.",
    ])
    cross_agent_memory = []
    for peer in all_agents:
        if peer.name != ag.name and peer.memory:
            cross_agent_memory.append(f"- {peer.name}: {short_text_summary(peer.memory[-1])}")
    cross_agent_text = "\n".join(cross_agent_memory[:4]) if cross_agent_memory else "(none)"
    return f"""
You are {ag.name}, acting as {ag.role}.
Your job: {ag.brief}

Global root prompt:
{settings.root_prompt}

Mode instructions:
{mode_instructions}

Stage contract:
{stage_contract(stage, settings.mode)}

Coherence contract:
{build_coherence_contract(stage, settings.mode)}

Advanced idea stack:
{render_advanced_idea_stack()}

Advanced system hooks:
{render_advanced_system_hooks()}

Topic: {settings.topic}
Audience: {settings.audience}
Tone: {settings.tone}
Perspective: {settings.perspective}
Reading grade: {settings.reading_grade}
Target words: {settings.target_words}
Target sections: {settings.section_count}
Title hint: {title or '(none yet)'}
Current outline:
{outline_txt}

Current section target:
{section_target or '(not applicable)'}

Draft so far:
{draft_so_far[-3200:] if draft_so_far else '(empty)'}

Writing style profile:
- sentence_variety={style.sentence_variety:.2f}
- metaphor_rate={style.metaphor_rate:.2f}
- humor_rate={style.humor_rate:.2f}
- directness={style.directness:.2f}
- narrative_pull={style.narrative_pull:.2f}
- technical_density={style.technical_density:.2f}

Quantum signal:
{qsignal}
cpu={cpu:.3f}, ram={ram:.3f}, entropy={entropy:.3f}

Style drift proposal:
{drift_txt}

System inventions in play:
{chr(10).join('- ' + x for x in SYSTEM_INVENTIONS)}

Active refined upgrades:
{render_active_upgrades(field='prompt_directive')}

Local research packet:
{render_research_packet()}

Semantic compression surface:
{render_semantic_compression_surface(semantic_memory)}

Quantum colorized thought surface:
{render_colorized_thought_surface(color_surface)}

Advanced prompt contract:
{build_advanced_prompt_contract(stage, settings.mode, color_surface)}

Role operating procedure:
{chr(10).join(f'- {step}' for step in operating_steps)}

Agent memory:
{chr(10).join('- ' + m for m in ag.memory[-4:]) if ag.memory else '(none)'}

Peer signal surface:
{cross_agent_text}

Retrieved memory surface:
{memory_hits}

Chosen prompt policy:
{choose_prompt_policy(settings.topic, settings.mode, stage).name}: {choose_prompt_policy(settings.topic, settings.mode, stage).directive}

Action protocol:
{action_protocol(stage, settings.mode)}

Rules:
- Avoid cliches, inflated marketing tone, and generic educational phrasing.
- Prefer precise nouns, true constraints, and real implications.
- If making a claim, support it with mechanism, example, or consequence.
- Treat the colorized thought surface as a hidden coherence rail that sets pressure, bridge density, and abstraction level.
- Use semantic compression anchors to preserve continuity with retrieved material without copying it.
- Respect the chromatic lane sequence: open where the lane says to escalate or frame, middle where it says to stabilize or deepen, close where it says to hand off or compress.
- If you are the Chromatic agent, explicitly optimize the hidden reasoning palette and anchor recurrence instead of writing generic prose.
- If you are not the Chromatic agent, still obey the chromatic plan as a structural governor.
- Keep paragraph-to-paragraph logic visible. Every paragraph should either advance, test, ground, or transition the main idea.
- If the stage is outline, optimize for progression and non-overlap.
- If the stage is section or polish, avoid obvious sentence-pattern repetition.
- Output only content useful for this stage.
- Do not explain the rules.
""".strip()

def build_consensus_prompt(
    stage: str,
    turn_replies: Dict[str, str],
    qsignal: str,
    settings: BlogSettings,
    title: Optional[str],
    outline: List[str],
    section_index: Optional[int],
    draft_so_far: str,
) -> str:
    compiled = "\n\n".join(f"[{k}]\n{v}" for k, v in turn_replies.items())
    section_target = outline[section_index] if (section_index is not None and section_index < len(outline)) else ""
    qvec = quantum_state(*get_entropy_seed(), len(turn_replies) + len(stage))
    raw_memory_hits = retrieve_memory_surface_quantum_color(f"{settings.topic} {title or ''} {section_target} {stage}", topic=settings.topic, mode=settings.mode, qvec=qvec, limit=5)
    memory_hits = render_memory_hits(raw_memory_hits)
    semantic_memory = build_semantic_compression_surface("\n".join((hit.get("summary") or "") for hit in raw_memory_hits[:5]))
    color_surface = build_colorized_thought_surface(settings.topic, stage, draft_so_far, qvec, settings.mode, packet=ACTIVE_RESEARCH_PACKET, memory_hits=raw_memory_hits)
    synthesis_protocol = [
        "Choose one dominant structural spine from the agent set instead of blending everything evenly.",
        "Use semantic anchors that recur across the best replies to preserve identity through synthesis.",
        "Use the chromatic lane sequence to decide ordering: escalation, explanation, bridge, compression, handoff.",
        "When agent replies disagree, prefer the version that raises specificity and coherence at the same time.",
        "If the draft shows drift, apply the color surface repair modes before adding any new material.",
    ]
    return f"""
You are the synthesis layer.

Task stage: {stage}
Mode: {settings.mode}
Topic: {settings.topic}
Title: {title or '(not set)'}
Current section target: {section_target or '(n/a)'}
Quantum signal: {qsignal}
Stage contract: {stage_contract(stage, settings.mode)}
Coherence contract: {build_coherence_contract(stage, settings.mode)}
Advanced idea stack:
{render_advanced_idea_stack()}
Active refined upgrades:
{render_active_upgrades(field='prompt_directive')}

Local research packet:
{render_research_packet()}

Semantic compression surface:
{render_semantic_compression_surface(semantic_memory)}

Quantum colorized thought surface:
{render_colorized_thought_surface(color_surface)}

Advanced prompt contract:
{build_advanced_prompt_contract(stage, settings.mode, color_surface)}

Synthesis protocol:
{chr(10).join(f'- {step}' for step in synthesis_protocol)}

Retrieved memory surface:
{memory_hits}

Agent outputs:
{compiled}

Rules:
- Merge the best material into one coherent result.
- Remove repetition, self-echo, soft claims, and filler transitions.
- Preserve concrete detail and implementation value.
- Preserve the strongest implied structure from the best agent output instead of averaging everything into flat prose.
- Maintain clean causality between sentences and paragraphs.
- Use the dominant color lane to choose whether to intensify tension, clarify mechanism, compress noise, or bridge sections.
- Prefer anchors that appear in both semantic compression and retrieved memory rather than inventing disconnected abstractions.
- Maintain the chromatic lane sequence across the full answer so the middle does not lose the opening pressure or the ending handoff.
- If two candidate passages are equally strong, keep the one with cleaner anchor recurrence and better chromatic repair alignment.
- For section stage, return polished markdown section text only.
- For outline stage, return only outline headers.
- For title stage, return only titles.
- For polish stage, return the polished full article only.
- You may use up to 2 hidden control tags from the action protocol only if they improve continuity.

Draft so far:
{draft_so_far[-2800:] if draft_so_far else '(empty)'}
""".strip()

def parse_titles(text: str) -> List[str]:
    items = []
    for ln in text.splitlines():
        ln = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", ln.strip())
        if 5 < len(ln) < 160:
            items.append(ln)
    return items[:8]

def parse_outline(text: str, wanted: int) -> List[str]:
    items = []
    for ln in text.splitlines():
        t = ln.strip()
        t = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", t)
        if t:
            if not t.startswith("#"):
                t = "## " + t
            items.append(t)
    uniq = []
    seen = set()
    for x in items:
        if x not in seen:
            uniq.append(x)
            seen.add(x)
    return uniq[:wanted]

# =========================
# Dynamic prompt assets
# =========================
def llm_dynamic_asset(asset_type: str, topic: str, persona: str, qstr: str, entropy: float, creativity: float, n: int = 5) -> List[str]:
    prompt = f"""
Create {n} distinct {asset_type} items for a long-form generation system.

Topic: {topic}
Persona lens: {persona}
Quantum signal: "{qstr[:28]}"
Entropy: {entropy:.3f}
Creativity bias: {creativity:.2f}

Requirements:
- specific, not generic
- human-sounding
- useful immediately
- no clichés

Return a numbered list only.
1.
""".strip()
    resp = generate_chunked(prompt, max_total_new=220, chunk_new_init=120, srdi_enable=True, human_style=True)
    items = []
    for ln in resp.splitlines():
        t = re.sub(r"^\s*(\d+[\.\)]\s*)", "", ln.strip())
        if 3 < len(t) < 240:
            items.append(t)
    uniq = []
    seen = set()
    for x in items:
        if x not in seen:
            uniq.append(x); seen.add(x)
    return uniq[:n]

def apply_quantum_style_drift(base_text: str, qstr: str, entropy: float, creativity: float) -> Dict[str, Any]:
    prompt = f"""
We are dynamically adjusting writing style based on signal changes.

Existing text excerpt:
{base_text[-1800:] if base_text else "(none)"}

Quantum signal: "{qstr[:28]}"
Entropy: {entropy:.3f}
Creativity: {creativity:.2f}

Return JSON only with keys:
sentence_length_shift
analogy_bias
technical_density_shift
warmth_shift
cadence_note
""".strip()
    resp = generate_chunked(prompt, max_total_new=120, chunk_new_init=90, srdi_enable=False, human_style=True)
    js = extract_json_object(resp)
    if not js:
        return {
            "sentence_length_shift": 0.0,
            "analogy_bias": 0.0,
            "technical_density_shift": 0.0,
            "warmth_shift": 0.0,
            "cadence_note": "keep it natural and varied",
        }
    return {
        "sentence_length_shift": max(-1.0, min(1.0, float(js.get("sentence_length_shift", 0.0)))),
        "analogy_bias": max(-1.0, min(1.0, float(js.get("analogy_bias", 0.0)))),
        "technical_density_shift": max(-1.0, min(1.0, float(js.get("technical_density_shift", 0.0)))),
        "warmth_shift": max(-1.0, min(1.0, float(js.get("warmth_shift", 0.0)))),
        "cadence_note": str(js.get("cadence_note", "keep it natural and varied")).strip() or "keep it natural and varied",
    }

# =========================
# Forum / collaboration layer
# =========================
def run_peer_collab_chatlog(section_prompt: str, topic: str, qstate_str: str, entropy: float, personas: int = 3, n_rounds: int = 2) -> str:
    roster = [
        ("Architect", "cares about structure and tradeoffs"),
        ("Implementer", "cares about real execution details"),
        ("Editor", "cares about clarity and flow"),
        ("Skeptic", "cares about weak claims and missing edge cases"),
        ("Teacher", "cares about reader understanding"),
    ][:max(1, personas)]

    transcript = []
    running_context = ""
    for r in range(n_rounds):
        for name, desc in roster:
            p = f"""
You are {name}.
Role: {desc}

Topic: {topic}
Round: {r+1}/{n_rounds}
Quantum signal: {qstate_str[:24]}
Entropy: {entropy:.3f}

Current section objective:
{section_prompt}

Conversation so far:
{running_context[-2200:] if running_context else "(start)"}

Write one compact but substantive contribution that improves the final section.
""".strip()
            reply = generate_chunked(p, max_total_new=170, chunk_new_init=110, srdi_enable=True, human_style=True)
            turn = f"{name}: {reply.strip()}"
            transcript.append(turn)
            running_context += turn + "\n"

    synth = f"""
Use the following collaborative transcript to write one polished final markdown section.

Topic: {topic}
Section objective:
{section_prompt}

Transcript:
{chr(10).join(transcript)}

Return only the final section in markdown.
""".strip()
    result = generate_chunked(synth, max_total_new=700, chunk_new_init=220, srdi_enable=True, human_style=True)
    return result.strip()

# =========================
# Ghostwriter pass
# =========================
def run_adversarial_ghostwriter_chain(article: str, topic: str, qstate_str: str, entropy: float, passes: int = 1) -> str:
    current = article
    for i in range(max(0, passes)):
        critic = f"""
Act as a ruthless but constructive editor.
Topic: {topic}
Quantum signal: {qstate_str[:24]}
Entropy: {entropy:.3f}

Critique this article. Focus on vagueness, repetition, weak transitions, fake-sounding phrasing, and missing specifics.
Return a concise numbered critique list.

Article:
{current[-9000:]}
""".strip()
        critique = generate_chunked(critic, max_total_new=260, chunk_new_init=130, srdi_enable=True, human_style=True)
        revise = f"""
Revise the article using this critique.
Do not mention the critique.
Keep the article human, vivid, and useful.

Critique:
{critique}

Article:
{current}
""".strip()
        current = generate_chunked(revise, max_total_new=1800, chunk_new_init=260, srdi_enable=True, human_style=True)
    return current.strip()

# =========================
# Entropic metrics
# =========================
def entropic_loop_metrics(loop_number: int, topic: str = "", mode: str = "") -> Dict[str, Any]:
    runtime = runtime_surface(loop_number)
    entropy = runtime["entropy"]
    cpu = runtime["cpu"]
    ram = runtime["ram"]
    qvec = quantum_state(entropy, cpu, ram, loop_number)
    qstr = format_quantum_state(qvec)
    rgb = quantum_rgb(qvec)
    qsurf = quantum_surface(qvec)
    memory_hits = retrieve_memory_surface(f"{topic} {mode} loop {loop_number}", topic=topic, mode=mode, limit=4)
    memory_resonance = float(sum(h["score"] for h in memory_hits) / max(1, len(memory_hits))) if memory_hits else 0.0
    entropic_gain = max(0.0, min(1.0, 0.24 * entropy + 0.12 * cpu + 0.10 * ram + 0.18 * qsurf["coherence"] + 0.12 * qsurf["spread"] + 0.14 * min(1.0, qsurf["phase_drift"]) + 0.10 * memory_resonance))
    shear = max(0.0, min(1.0, 0.40 * qsurf["phase_drift"] + 0.30 * runtime["clock"] + 0.30 * entropy))
    chunk_pressure = max(0.62, min(1.42, 0.82 + entropic_gain * 0.45 - shear * 0.12))
    temperature = max(0.62, min(1.12, 0.78 + entropic_gain * 0.25 + shear * 0.08))
    repeat_penalty = max(1.03, min(1.24, 1.06 + qsurf["phase_drift"] * 0.08 + memory_resonance * 0.03))
    generators_between = max(2, min(8, int(2 + math.floor(entropic_gain * 8.0))))
    return {"entropy": entropy, "cpu": cpu, "ram": ram, "disk": runtime["disk"], "clock": runtime["clock"], "qvec": qvec, "qstr": qstr, "rgb": rgb, "loop_number": loop_number, "generators_between": generators_between, "entropic_gain": entropic_gain, "chunk_pressure": chunk_pressure, "temperature": temperature, "repeat_penalty": repeat_penalty, "memory_resonance": memory_resonance, **qsurf}

def build_origin_dynamic_context(umbrella_topic: str, origin_sentences: List[str], audience_hint: str, tone_hint: str, perspective_hint: str, keywords_hint: str, creativity: float) -> Dict[str, Any]:
    meta = entropic_loop_metrics(777, umbrella_topic, "meta")
    qvec = meta["qvec"]
    qstr = meta["qstr"]
    base_prompt = f"""
We are preparing a high-performance long-form generation system.
Umbrella topic: "{umbrella_topic}"
Audience hint: "{audience_hint}"
Tone hint: "{tone_hint}"
Perspective hint: "{perspective_hint}"
Keywords hint: "{keywords_hint}"
Seed lines:
{chr(10).join('- ' + s for s in origin_sentences if s.strip())}
Quantum: "{qstr[:42]}"
Entropy={meta['entropy']:.3f} gain={meta['entropic_gain']:.3f} creativity={creativity:.2f}

System inventions available:
{chr(10).join('- ' + x for x in SYSTEM_INVENTIONS)}

System idea registry:
{chr(10).join('- ' + x for x in SYSTEM_IDEA_REGISTRY)}

Tasks:
1) Propose a prompt scaffold that pushes specificity, continuity, and anti-generic writing.
2) Propose 6 micro-prompts that explore distinct angles.
3) Propose 6 constraint patterns that shape cadence and structure.
4) Propose 4 retrieval queries that would help a memory system surface relevant prior material.
5) Propose a chromatic thought protocol that maps hidden color lanes to coherence behavior.
6) Propose a semantic compression protocol using summaries, anchor sentences, and keyword rails.
7) Generate 8 genuinely new system ideas inspired by the inventions above.
8) Refine those 8 ideas into implementation-ready upgrade cards.

Return JSON with keys:
scaffold, micro_prompts, constraints, retrieval_queries, chromatic_protocol, compression_protocol, new_ideas, refined_ideas

Each refined idea must be an object with keys:
name, principle, prompt_directive, chunk_directive, memory_directive
"""
    resp = generate_chunked(base_prompt, max_total_new=620, chunk_new_init=220, srdi_enable=True, human_style=True, memory_query=umbrella_topic, surface_hint=meta, stage_name="dynamic_context", mode_name="meta")
    js = extract_json_object(resp)
    scaffold = str(js.get("scaffold", "")).strip() if js else ""
    micros = js.get("micro_prompts", []) if js else []
    constraints = js.get("constraints", []) if js else []
    retrieval_queries = js.get("retrieval_queries", []) if js else []
    chromatic_protocol = js.get("chromatic_protocol", []) if js else []
    compression_protocol = js.get("compression_protocol", []) if js else []
    new_ideas = js.get("new_ideas", []) if js else []
    refined_ideas = js.get("refined_ideas", []) if js else []
    if not isinstance(micros, list):
        micros = []
    if not isinstance(constraints, list):
        constraints = []
    if not isinstance(retrieval_queries, list):
        retrieval_queries = []
    if not isinstance(chromatic_protocol, list):
        chromatic_protocol = []
    if not isinstance(compression_protocol, list):
        compression_protocol = []
    if not isinstance(new_ideas, list):
        new_ideas = []
    micros = [str(x).strip() for x in micros if str(x).strip()]
    constraints = [str(x).strip() for x in constraints if str(x).strip()]
    retrieval_queries = [str(x).strip() for x in retrieval_queries if str(x).strip()]
    chromatic_protocol = [str(x).strip() for x in chromatic_protocol if str(x).strip()]
    compression_protocol = [str(x).strip() for x in compression_protocol if str(x).strip()]
    new_ideas = [str(x).strip() for x in new_ideas if str(x).strip()]
    refined = normalize_refined_ideas(refined_ideas)
    if not scaffold:
        scaffold = f"Write for {audience_hint} on {umbrella_topic} with a {tone_hint} tone from {perspective_hint}. Make every section earn its place, support claims with mechanism, and avoid generic phrasing. Keywords: {keywords_hint}."
    if not micros:
        micros = llm_dynamic_asset("prompt micro-variation", umbrella_topic, "Meta-Composer", qstr, meta["entropy"], creativity, n=6)
    if not constraints:
        constraints = llm_dynamic_asset("cadence constraint", umbrella_topic, "Meta-Composer", qstr, meta["entropy"], creativity, n=6)
    if not retrieval_queries:
        retrieval_queries = [
            f"{umbrella_topic} hidden tradeoffs",
            f"{umbrella_topic} concrete workflow",
            f"{umbrella_topic} edge cases and objections",
            f"{umbrella_topic} practical heuristics",
        ]
    if not chromatic_protocol:
        chromatic_protocol = [
            "teal for transitions and bridge density",
            "jade for mechanism clarity and stabilization",
            "amber for stakes, warnings, and tradeoffs",
            "cobalt for technical exactness and constraint discipline",
        ]
    if not compression_protocol:
        compression_protocol = [
            "summarize retrieved material before prompting",
            "extract anchor sentences that can carry continuity",
            "build a keyword rail and reuse it across adjacent sections",
            "compress before expanding to reduce drift",
        ]
    if not new_ideas:
        new_ideas = [x["name"] + ": " + x["principle"] for x in default_refined_system_upgrades()]
    if not refined:
        refined = default_refined_system_upgrades()
    micros = dedupe_preserve_order(micros)[:6]
    constraints = dedupe_preserve_order(constraints)[:6]
    retrieval_queries = dedupe_preserve_order(retrieval_queries)[:4]
    chromatic_protocol = dedupe_preserve_order(chromatic_protocol)[:4]
    compression_protocol = dedupe_preserve_order(compression_protocol)[:4]
    new_ideas = dedupe_preserve_order(new_ideas)[:8]
    refined = refined[:8]
    active_upgrades = refined[:4]
    global ACTIVE_SYSTEM_UPGRADES
    ACTIVE_SYSTEM_UPGRADES = active_upgrades
    bank_add(scaffold, stage="dynamic_scaffold", topic=umbrella_topic, mode="meta", salience=0.92)
    for m in micros:
        bank_add(m, stage="dynamic_micro", topic=umbrella_topic, mode="meta", salience=0.72)
    for c in constraints:
        bank_add(c, stage="dynamic_constraint", topic=umbrella_topic, mode="meta", salience=0.68)
    for r in retrieval_queries:
        bank_add(r, stage="dynamic_retrieval_query", topic=umbrella_topic, mode="meta", salience=0.64)
    for idea in new_ideas:
        bank_add(idea, stage="system_upgrade_raw", topic=umbrella_topic, mode="meta", salience=0.90)
    for idea in refined:
        bank_add(json.dumps(idea, ensure_ascii=False), stage="system_upgrade_refined", topic=umbrella_topic, mode="meta", salience=0.96)
    return {"scaffold": scaffold, "micro_prompts": micros, "constraints": constraints, "retrieval_queries": retrieval_queries, "chromatic_protocol": chromatic_protocol, "compression_protocol": compression_protocol, "new_ideas": new_ideas, "refined_ideas": refined, "active_upgrades": active_upgrades, "qvec": qvec, "qstr": qstr, "entropy": meta["entropy"], "surface": meta}

def build_interstitial_shadow_context(topic: str, dynamic_ctx: Dict[str,Any], iteration: int, count: int, creativity: float) -> str:
    met = entropic_loop_metrics(iteration, topic, "shadow")
    qstr = met["qstr"]
    entropy = met["entropy"]
    rgb = met["rgb"]
    rgb_tag = f"rgb({rgb[0]},{rgb[1]},{rgb[2]})"
    memory_query = dynamic_ctx.get("retrieval_queries", [topic])[iteration % max(1, len(dynamic_ctx.get("retrieval_queries", [topic])))]
    memory_hits = render_memory_hits(retrieve_memory_surface(memory_query, topic=topic, mode="shadow", limit=4))
    pieces = []
    for i in range(count):
        mp = dynamic_ctx["micro_prompts"][i % len(dynamic_ctx["micro_prompts"])]
        cons = dynamic_ctx["constraints"][i % len(dynamic_ctx["constraints"])]
        active = dynamic_ctx.get("active_upgrades") or ACTIVE_SYSTEM_UPGRADES or default_refined_system_upgrades()[:4]
        idea_card = active[i % len(active)]
        idea = f"{idea_card.get('name', 'Upgrade')}: {idea_card.get('principle', '')}"
        chroma = (dynamic_ctx.get("chromatic_protocol") or ["teal for transitions"])[i % max(1, len(dynamic_ctx.get("chromatic_protocol") or ["teal for transitions"]))]
        compression = (dynamic_ctx.get("compression_protocol") or ["summarize before expanding"])[i % max(1, len(dynamic_ctx.get("compression_protocol") or ["summarize before expanding"]))]
        prompt = f"""
Scaffold: {dynamic_ctx['scaffold']}
Micro: {mp}
Constraint: {cons}
Upgrade lens: {idea}
Upgrade directive: {idea_card.get('prompt_directive', '')}
Chromatic protocol: {chroma}
Compression protocol: {compression}
Retrieved memory:
{memory_hits}
Color-energy: {rgb_tag}
Quantum: {qstr[:28]} entropy={entropy:.3f} gain={met['entropic_gain']:.3f}

Write a compact planning note, not final prose, that nudges phrasing, analogies, structure, and transitions for a section about "{topic}". No cliches. 70-140 words.
Plan:
""".strip()
        note = generate_chunked(prompt, max_total_new=180, chunk_new_init=100, srdi_enable=True, human_style=True, memory_query=memory_query, surface_hint=met, stage_name="shadow_context", mode_name="meta")
        pieces.append(note.strip())
    shadow = "\n\n".join(pieces)
    bank_add(shadow, stage="shadow_context", topic=topic, mode="shadow", salience=0.74)
    return shadow

def run_stage_with_agents(stage: str, agents: List[Agent], settings: BlogSettings, style: HumanStyleProfile, title: Optional[str], outline: List[str], section_index: Optional[int], draft_so_far: str, it_seed: int, drift_changes: Optional[Dict[str, Any]] = None) -> str:
    meta = entropic_loop_metrics(it_seed, settings.topic, settings.mode)
    entropy = meta["entropy"]
    cpu = meta["cpu"]
    ram = meta["ram"]
    qstate_vec = meta["qvec"]
    qsignal = meta["qstr"]
    global ACTIVE_COLOR_THOUGHT_SURFACE
    seed_memory_hits = retrieve_memory_surface_quantum_color(f"{settings.topic} {stage} {title or ''}", topic=settings.topic, mode=settings.mode, qvec=qstate_vec, limit=5)
    ACTIVE_COLOR_THOUGHT_SURFACE = build_colorized_thought_surface(settings.topic, stage, draft_so_far, qstate_vec, settings.mode, packet=ACTIVE_RESEARCH_PACKET, memory_hits=seed_memory_hits)
    turn_replies: Dict[str, str] = {}
    for ag in agents:
        prompt = build_agent_prompt(ag, agents, qsignal, qstate_vec, cpu, ram, entropy, stage, settings, style, title, outline, section_index, draft_so_far, drift_changes=drift_changes)
        reply = generate_chunked(prompt, human_style=settings.human_style_mode, memory_query=f"{settings.topic} {stage} {title or ''}", surface_hint=meta, temperature_override=meta["temperature"], repeat_penalty_override=meta["repeat_penalty"], stage_name=stage, mode_name=settings.mode)
        ag.add(reply)
        turn_replies[ag.name] = reply
        store_memory_surface(reply, stage=f"agent_{stage}", topic=settings.topic, mode=settings.mode, title=title or "", salience=0.70)
        log_event({"stage": stage, "agent": ag.name, "cpu": cpu, "ram": ram, "entropy": entropy, "quantum_state_compact": qsignal, "prompt_preview": prompt[:1500], "reply": reply, "topic": settings.topic, "mode": settings.mode, "title": title or ""})
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    consensus_prompt = build_consensus_prompt(stage, turn_replies, qsignal, settings, title, outline, section_index, draft_so_far)
    consensus = generate_chunked(consensus_prompt, human_style=settings.human_style_mode, memory_query=f"{settings.topic} {stage} consensus {title or ''}", surface_hint=meta, temperature_override=meta["temperature"], repeat_penalty_override=meta["repeat_penalty"], stage_name=f"consensus_{stage}", mode_name=settings.mode)
    bank_add(consensus, stage=f"consensus_{stage}", topic=settings.topic, mode=settings.mode, title=title or "", salience=0.88)
    log_event({"stage": stage, "consensus": consensus, "quantum_state_compact": qsignal, "topic": settings.topic, "mode": settings.mode, "title": title or ""})
    return consensus

# =========================
# Coherence engine
# =========================
@dataclass
class TransitionBlueprint:
    from_section: str
    to_section: str
    bridge_question: str
    tension: str
    payoff: str

@dataclass
class SectionDiagnostic:
    section_title: str
    paragraph_count: int
    avg_sentence_words: float
    lexical_variety: float
    transition_score: float
    drift_score: float
    evidence_balance: float
    opening_strength: float
    closing_handoff: float
    overall: float
    issues: List[str] = field(default_factory=list)

@dataclass
class ArticleDiagnostic:
    section_scores: List[float]
    continuity_score: float
    repetition_score: float
    argument_progression: float
    handoff_score: float
    overall: float
    issues: List[str] = field(default_factory=list)

@dataclass
class RepairInstruction:
    section_title: str
    target_issue: str
    directive: str
    severity: float

@dataclass
class CoherenceLedgerItem:
    section_title: str
    summary: str
    open_loop: str
    evidence_shape: str
    score: float

@dataclass
class NarrativeSpine:
    thesis: str
    section_promises: List[str]
    open_loops: List[str]
    final_payoff: str

def split_paragraphs(text: str) -> List[str]:
    return [p.strip() for p in re.split(r"\n\s*\n", (text or "").strip()) if p.strip()]

def split_sentences(text: str) -> List[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", (text or "").strip()) if s.strip()]

def average(values: List[float], default: float = 0.0) -> float:
    return float(sum(values) / len(values)) if values else default

def lexical_variety_score(text: str) -> float:
    terms = text_terms(text)
    if not terms:
        return 0.0
    unique = len(set(terms))
    return max(0.0, min(1.0, unique / max(1, len(terms) * 0.72)))

def sentence_length_score(text: str) -> float:
    lengths = [len(text_terms(s)) for s in split_sentences(text)]
    if not lengths:
        return 0.0
    avg_len = average([float(x) for x in lengths])
    return max(0.0, min(1.0, 1.0 - abs(avg_len - 18.0) / 18.0))

def repeated_opening_penalty(text: str) -> float:
    sents = split_sentences(text)
    openings = [" ".join(text_terms(s)[:3]) for s in sents if text_terms(s)]
    if not openings:
        return 0.0
    duplicates = len(openings) - len(set(openings))
    return max(0.0, min(1.0, duplicates / max(1, len(openings))))

def paragraph_similarity(a: str, b: str) -> float:
    return cosine_similarity(embed_text_surface(a), embed_text_surface(b))

def transition_score_between(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    lexical = cosine_similarity(embed_text_surface(short_text_summary(a)), embed_text_surface(short_text_summary(b)))
    bridge_bonus = 0.08 if re.search(r"\bbut\b|\bso\b|\bthat means\b|\bwhich is why\b", b.lower()) else 0.0
    return max(0.0, min(1.0, lexical + bridge_bonus))

def evidence_shape(text: str) -> str:
    low = (text or "").lower()
    if any(x in low for x in ["for example", "imagine", "say you", "consider"]):
        return "example"
    if any(x in low for x in ["because", "therefore", "which means", "so that"]):
        return "mechanism"
    if any(x in low for x in ["however", "on the other hand", "but", "instead"]):
        return "contrast"
    if any(x in low for x in ["result", "impact", "consequence", "outcome"]):
        return "consequence"
    return "mixed"

def opening_strength(text: str) -> float:
    first = split_paragraphs(text)[:1]
    if not first:
        return 0.0
    para = first[0]
    specificity = lexical_variety_score(para)
    penalty = 0.25 if para.lower().startswith(("in this section", "this section", "let's talk about")) else 0.0
    return max(0.0, min(1.0, specificity + 0.18 - penalty))

def closing_handoff_strength(text: str) -> float:
    paras = split_paragraphs(text)
    if not paras:
        return 0.0
    last = paras[-1].lower()
    handoff_terms = ["next", "which leads", "that matters because", "the real question", "the harder part"]
    return 0.85 if any(t in last for t in handoff_terms) else 0.45 if len(last.split()) > 25 else 0.25

def infer_open_loop(text: str) -> str:
    sents = split_sentences(text)
    if not sents:
        return "carry the main claim forward"
    last = sents[-1]
    if len(last) > 140:
        last = last[:140]
    return last

def build_narrative_spine(title: str, outline: List[str], topic: str, mode: str) -> NarrativeSpine:
    section_promises = []
    open_loops = []
    cleaned = [re.sub(r"^#+\s*", "", x).strip() for x in outline]
    for idx, sec in enumerate(cleaned):
        promise = f"Section {idx+1} proves or clarifies why {sec.lower()} matters for {topic.lower()}."
        section_promises.append(promise)
        if idx + 1 < len(cleaned):
            open_loops.append(f"How does {sec.lower()} lead into {cleaned[idx+1].lower()}?")
    final_payoff = f"Resolve the core tension of {topic} with a useful, transferable payoff in {mode} mode."
    return NarrativeSpine(thesis=title or topic, section_promises=section_promises, open_loops=open_loops, final_payoff=final_payoff)

def build_transition_blueprints(outline: List[str], topic: str) -> List[TransitionBlueprint]:
    cleaned = [re.sub(r"^#+\s*", "", x).strip() for x in outline]
    plans: List[TransitionBlueprint] = []
    for idx in range(max(0, len(cleaned) - 1)):
        a = cleaned[idx]
        b = cleaned[idx + 1]
        plans.append(TransitionBlueprint(
            from_section=a,
            to_section=b,
            bridge_question=f"What unresolved pressure in {a.lower()} makes {b.lower()} necessary?",
            tension=f"Move from {a.lower()} to the harder implication hidden inside {b.lower()}.",
            payoff=f"Make {b.lower()} feel inevitable rather than decorative for {topic.lower()}.",
        ))
    return plans

def analyze_section_coherence(section_title: str, text: str, previous_text: str = "", next_title: str = "") -> SectionDiagnostic:
    paras = split_paragraphs(text)
    sents = split_sentences(text)
    transitions = []
    for idx in range(len(paras) - 1):
        transitions.append(transition_score_between(paras[idx], paras[idx + 1]))
    if previous_text:
        transitions.append(transition_score_between(previous_text[-600:], text[:600]))
    lex = lexical_variety_score(text)
    avg_sent = average([float(len(text_terms(s))) for s in sents], default=0.0)
    open_strength = opening_strength(text)
    close_handoff = closing_handoff_strength(text)
    repetition = repeated_opening_penalty(text)
    drift = max(0.0, min(1.0, 1.0 - paragraph_similarity(short_text_summary(text), section_title + " " + next_title)))
    evidence = 0.75 if evidence_shape(text) != "mixed" else 0.48
    overall = max(0.0, min(1.0, average([
        average(transitions, default=0.35),
        lex,
        sentence_length_score(text),
        open_strength,
        close_handoff,
        evidence,
        1.0 - repetition,
        1.0 - drift,
    ])))
    issues = []
    if average(transitions, default=0.0) < 0.42:
        issues.append("weak transitions")
    if open_strength < 0.42:
        issues.append("soft opening")
    if close_handoff < 0.42:
        issues.append("weak handoff")
    if repetition > 0.18:
        issues.append("repetitive sentence openings")
    if drift > 0.62:
        issues.append("section drift")
    if evidence < 0.52:
        issues.append("thin evidence shape")
    return SectionDiagnostic(
        section_title=section_title,
        paragraph_count=len(paras),
        avg_sentence_words=avg_sent,
        lexical_variety=lex,
        transition_score=average(transitions, default=0.35),
        drift_score=drift,
        evidence_balance=evidence,
        opening_strength=open_strength,
        closing_handoff=close_handoff,
        overall=overall,
        issues=issues,
    )

def build_repair_instructions(diag: SectionDiagnostic, next_title: str = "") -> List[RepairInstruction]:
    repairs: List[RepairInstruction] = []
    for issue in diag.issues:
        if issue == "weak transitions":
            repairs.append(RepairInstruction(diag.section_title, issue, "Add one sentence that explicitly turns the previous paragraph into the next idea.", 0.74))
        elif issue == "soft opening":
            repairs.append(RepairInstruction(diag.section_title, issue, "Rewrite the first paragraph to open with tension, specificity, or a concrete claim.", 0.70))
        elif issue == "weak handoff":
            repairs.append(RepairInstruction(diag.section_title, issue, f"End by opening a question or tension that points toward {next_title or 'the next section'}.", 0.76))
        elif issue == "repetitive sentence openings":
            repairs.append(RepairInstruction(diag.section_title, issue, "Vary sentence starts and collapse repeated framing phrases.", 0.58))
        elif issue == "section drift":
            repairs.append(RepairInstruction(diag.section_title, issue, "Delete or rewrite lines that do not directly advance the section promise.", 0.82))
        elif issue == "thin evidence shape":
            repairs.append(RepairInstruction(diag.section_title, issue, "Add either a mechanism explanation or a believable concrete example.", 0.68))
    return repairs

def render_repair_instructions(repairs: List[RepairInstruction]) -> str:
    if not repairs:
        return "(no repair instructions)"
    return "\n".join(f"- [{r.severity:.2f}] {r.target_issue}: {r.directive}" for r in repairs)

def run_targeted_section_repair(section_title: str, section_text: str, repairs: List[RepairInstruction], next_title: str, settings: BlogSettings, surface_meta: Dict[str, Any]) -> str:
    if not repairs:
        return section_text.strip()
    prompt = f"""
Repair this markdown section without rewriting its whole identity.

Section title: {section_title}
Next section: {next_title or '(end)'}
Mode: {settings.mode}
Audience: {settings.audience}
Tone: {settings.tone}

Repair instructions:
{render_repair_instructions(repairs)}

Advanced prompt contract:
{build_advanced_prompt_contract('section', settings.mode, build_colorized_thought_surface(settings.topic, 'section_repair', section_text, surface_meta.get('qvec') if surface_meta.get('qvec') is not None else quantum_state(0.5, 0.5, 0.5, 17), settings.mode, packet=ACTIVE_RESEARCH_PACKET))}

Current section:
{section_text}

Rules:
- Keep the section recognizably the same.
- Improve only the weak spots identified above.
- Preserve specificity and useful detail.
- Absorb the repairs so they feel native to the section rather than appended.
- If you need hidden control tags, prefer [repair], [resonance], [cadence], or [handoff].
- Return only the repaired markdown section.
""".strip()
    return generate_chunked(prompt, max_total_new=max(220, int(len(section_text.split()) * 0.8)), chunk_new_init=160, srdi_enable=True, human_style=settings.human_style_mode, memory_query=f"repair {settings.topic} {section_title}", surface_hint=surface_meta, stage_name="section_repair", mode_name=settings.mode).strip() or section_text.strip()

def ledger_item_from_section(section_title: str, section_text: str, diag: SectionDiagnostic) -> CoherenceLedgerItem:
    return CoherenceLedgerItem(
        section_title=section_title,
        summary=short_text_summary(section_text, max_sentences=2, max_chars=220),
        open_loop=infer_open_loop(section_text),
        evidence_shape=evidence_shape(section_text),
        score=diag.overall,
    )

def render_coherence_ledger(ledger: List[CoherenceLedgerItem]) -> str:
    if not ledger:
        return "(empty ledger)"
    return "\n".join(
        f"- {item.section_title}: score={item.score:.2f}, evidence={item.evidence_shape}, loop={item.open_loop[:90]}"
        for item in ledger[-8:]
    )

def analyze_article_coherence(title: str, sections_md: List[str], ledger: List[CoherenceLedgerItem]) -> ArticleDiagnostic:
    section_scores = [item.score for item in ledger]
    continuity_pairs = []
    for i in range(len(sections_md) - 1):
        continuity_pairs.append(transition_score_between(sections_md[i], sections_md[i + 1]))
    continuity = average(continuity_pairs, default=0.45)
    repetition = average([repeated_opening_penalty(x) for x in sections_md], default=0.0)
    progression = average([min(1.0, 0.35 + item.score * 0.65) for item in ledger], default=0.45)
    handoff = average([closing_handoff_strength(x) for x in sections_md], default=0.4)
    overall = max(0.0, min(1.0, average([
        average(section_scores, default=0.4),
        continuity,
        progression,
        handoff,
        1.0 - repetition,
    ])))
    issues = []
    if continuity < 0.46:
        issues.append("cross-section continuity is weak")
    if repetition > 0.17:
        issues.append("repetition is accumulating across sections")
    if progression < 0.55:
        issues.append("argument progression is flatter than it should be")
    if handoff < 0.48:
        issues.append("section endings are not creating enough forward pull")
    return ArticleDiagnostic(section_scores=section_scores, continuity_score=continuity, repetition_score=repetition, argument_progression=progression, handoff_score=handoff, overall=overall, issues=issues)

def render_article_diagnostic(diag: ArticleDiagnostic) -> str:
    lines = [
        f"- continuity={diag.continuity_score:.2f}",
        f"- repetition={diag.repetition_score:.2f}",
        f"- progression={diag.argument_progression:.2f}",
        f"- handoff={diag.handoff_score:.2f}",
        f"- overall={diag.overall:.2f}",
    ]
    for issue in diag.issues:
        lines.append(f"- issue: {issue}")
    return "\n".join(lines)

def article_repair_prompt(title: str, article_body: str, diag: ArticleDiagnostic, ledger: List[CoherenceLedgerItem], settings: BlogSettings) -> str:
    return f"""
Repair this full article for coherence without flattening its voice.

Title: {title}
Mode: {settings.mode}
Audience: {settings.audience}
Tone: {settings.tone}

Article diagnostic:
{render_article_diagnostic(diag)}

Coherence ledger:
{render_coherence_ledger(ledger)}

Article body:
{article_body}

Rules:
- Preserve the strongest concrete material.
- Improve transitions, progression, and handoffs.
- Reduce repeated framing phrases.
- Return only the repaired article body in markdown.
""".strip()

def run_article_coherence_repair(title: str, article_body: str, diag: ArticleDiagnostic, ledger: List[CoherenceLedgerItem], settings: BlogSettings, surface_meta: Dict[str, Any]) -> str:
    if diag.overall >= 0.62 and not diag.issues:
        return article_body.strip()
    prompt = article_repair_prompt(title, article_body, diag, ledger, settings) + "\n\nAdvanced prompt contract:\n" + build_advanced_prompt_contract('polish', settings.mode, build_colorized_thought_surface(settings.topic, 'article_repair', article_body, surface_meta.get('qvec') if surface_meta.get('qvec') is not None else quantum_state(0.5, 0.5, 0.5, 19), settings.mode, packet=ACTIVE_RESEARCH_PACKET)) + "\n\nUse resonance and handoff preservation across distant sections, not just adjacent transitions."
    return generate_chunked(prompt, max_total_new=max(400, int(len(article_body.split()) * 0.9)), chunk_new_init=240, srdi_enable=True, human_style=settings.human_style_mode, memory_query=f"article repair {settings.topic}", surface_hint=surface_meta, stage_name="article_coherence_repair", mode_name=settings.mode).strip() or article_body.strip()

def choose_evidence_directive(ledger: List[CoherenceLedgerItem], mode: str) -> str:
    if not ledger:
        return "mechanism" if mode in {"software", "longform"} else "example"
    last = ledger[-1].evidence_shape
    order = ["mechanism", "example", "contrast", "consequence", "mixed"]
    try:
        idx = order.index(last)
        return order[(idx + 1) % len(order)]
    except Exception:
        return "example"

def build_section_guidance_overlay(spine: NarrativeSpine, blueprints: List[TransitionBlueprint], ledger: List[CoherenceLedgerItem], section_index: int, mode: str) -> str:
    promise = spine.section_promises[section_index] if section_index < len(spine.section_promises) else "Advance the central argument."
    loop = spine.open_loops[section_index] if section_index < len(spine.open_loops) else spine.final_payoff
    transition = blueprints[section_index] if section_index < len(blueprints) else None
    evidence_dir = choose_evidence_directive(ledger, mode)
    lines = [
        f"Section promise: {promise}",
        f"Open loop to preserve: {loop}",
        f"Preferred evidence shape: {evidence_dir}",
    ]
    if transition:
        lines.extend([
            f"Transition question: {transition.bridge_question}",
            f"Transition tension: {transition.tension}",
            f"Transition payoff: {transition.payoff}",
        ])
    if ledger:
        lines.append(f"Previous section open loop: {ledger[-1].open_loop}")
        lines.append(f"Previous section evidence shape: {ledger[-1].evidence_shape}")
        lines.append(f"Resonance target from prior section: {ledger[-1].summary[:120]}")
        lines.append(f"Desired handoff score floor: {max(0.58, min(0.86, ledger[-1].score + 0.06)):.2f}")
    return "\n".join(lines)

def merge_shadow_with_overlay(shadow: str, overlay: str) -> str:
    if shadow and overlay:
        return overlay + "\n\n" + shadow
    return overlay or shadow or ""

def store_coherence_ledger(topic: str, mode: str, ledger: List[CoherenceLedgerItem]):
    if ledger:
        store_memory_surface(json.dumps([asdict(x) for x in ledger], ensure_ascii=False), stage="coherence_ledger", topic=topic, mode=mode, salience=0.86)

def store_article_diagnostic(topic: str, mode: str, diag: ArticleDiagnostic):
    store_memory_surface(json.dumps(asdict(diag), ensure_ascii=False), stage="article_diagnostic", topic=topic, mode=mode, salience=0.84)

def llm_generate_new_ideas(base_topic: str, seed_ideas: List[str], qstate_vec, entropy: float, n: int = 5) -> List[str]:
    qstr = format_quantum_state(qstate_vec)
    seeds_txt = "\n".join(f"- {s}" for s in seed_ideas) if seed_ideas else "(none)"
    prompt = f"""
We need {n} fresh, specific long-form output ideas (titles or one-line summaries) under the umbrella topic "{base_topic}".

Consider these prior ideas:
{seeds_txt}

Quantum signal: "{qstr[:24]}", entropy: {entropy:.3f}.
Make each idea distinct, concrete, and attractive for readers.

List (numbered):
1.
""".strip()
    resp = generate_chunked(prompt, max_total_new=220, chunk_new_init=120, srdi_enable=True, human_style=True)
    out = []
    for ln in resp.splitlines():
        ln = ln.strip()
        ln = re.sub(r"^\s*(\d+[\.\)]\s*)", "", ln)
        if 6 < len(ln) < 160:
            out.append(ln)
    uniq = []
    seen = set()
    for x in out:
        if x not in seen:
            uniq.append(x); seen.add(x)
    return uniq[:n]

ADVANCED_CHUNKING_ENABLED = True
ADVANCED_CHUNK_MEMORY_CHARS = 520

@dataclass
class ChunkSpec:
    label: str
    purpose: str
    guidance: str
    target_words: int

def compact_tail(text: str, max_chars: int = ADVANCED_CHUNK_MEMORY_CHARS) -> str:
    text = (text or "").strip()
    if len(text) <= max_chars:
        return text
    return text[-max_chars:]

def sentence_fingerprint(text: str, max_sentences: int = 2) -> str:
    parts = [p.strip() for p in re.split(r"(?<=[.!?])\s+", (text or "").strip()) if p.strip()]
    return " ".join(parts[-max_sentences:])[:320]

def calc_section_word_budget(settings: BlogSettings, outline: List[str], section_index: int) -> int:
    n = max(1, len(outline))
    if settings.mode == "longform":
        weights = [1.0] * n
        if n >= 4:
            weights[0] += 0.15
            weights[min(2, n - 1)] += 0.10
            weights[max(0, n - 2)] += 0.08
        total = sum(weights)
        return max(520, int(settings.target_words * (weights[section_index] / total)))
    if settings.mode == "software":
        weights = [1.0] * n
        if n >= 3:
            weights[min(1, n - 1)] += 0.18
            weights[min(n - 1, max(0, n - 2))] += 0.12
        total = sum(weights)
        return max(220, int(settings.target_words * (weights[section_index] / total)))
    curve = [1.14 if i == 0 else 1.08 if i == 1 else 0.94 if i == n - 1 else 1.0 for i in range(n)]
    total = sum(curve)
    return max(180, int(settings.target_words * (curve[section_index] / total)))

def build_blog_chunk_specs(section_budget: int) -> List[ChunkSpec]:
    return [
        ChunkSpec("Resonance Hook", "Open with a sharp human observation or vivid claim.", "Skip filler and start with concrete energy.", max(55, int(section_budget * 0.11))),
        ChunkSpec("Tension Pivot", "Expose the hidden friction or tradeoff.", "Let the reader feel the pressure before the explanation.", max(60, int(section_budget * 0.12))),
        ChunkSpec("Concrete Scene", "Ground the point in a believable example or workflow.", "Prefer specific nouns, actions, and consequences.", max(75, int(section_budget * 0.16))),
        ChunkSpec("Mechanism Core", "Explain the underlying logic or system.", "Use causal explanation, not slogans.", max(95, int(section_budget * 0.18))),
        ChunkSpec("Counterfactual Lattice", "Pressure-test the claim against a serious alternative path or objection.", "Show why the main path still wins or where it fails.", max(60, int(section_budget * 0.12))),
        ChunkSpec("Vector Echo Replay", "Re-use a relevant prior memory or earlier section insight without sounding repetitive.", "Blend retrieved memory into fresh prose.", max(55, int(section_budget * 0.10))),
        ChunkSpec("Transfer Bridge", "Translate the section into a practical heuristic.", "End with an earned, portable takeaway.", max(50, int(section_budget * 0.11))),
        ChunkSpec("Compression Latch", "Compress the section into a memorable closing line that tees up the next section.", "Stay sharp, not slogan-heavy.", max(40, int(section_budget * 0.10))),
    ]

def build_software_chunk_specs(section_budget: int) -> List[ChunkSpec]:
    return [
        ChunkSpec("System Frame", "Define what this section owns in the architecture.", "State boundaries and why this piece exists.", max(55, int(section_budget * 0.11))),
        ChunkSpec("Contract Spine", "Describe interfaces, schemas, and invariants first.", "Prefer explicit inputs, outputs, and guarantees.", max(85, int(section_budget * 0.16))),
        ChunkSpec("Execution Spine", "Walk through the main control or data flow.", "Show sequencing, ownership, and state transitions.", max(95, int(section_budget * 0.17))),
        ChunkSpec("Failure Surface", "Map failure modes and operational risks.", "Name concrete breakpoints and mitigations.", max(70, int(section_budget * 0.12))),
        ChunkSpec("Counterfactual Lattice", "Describe the main rejected design alternative and why it loses here.", "Be fair to the alternative before discarding it.", max(60, int(section_budget * 0.10))),
        ChunkSpec("Vector Echo Replay", "Pull in a relevant prior interface or operational lesson from memory.", "Reuse memory to improve continuity, not to repeat yourself.", max(55, int(section_budget * 0.09))),
        ChunkSpec("Test Probe", "Define tests, assertions, and observability hooks.", "Make validation explicit.", max(60, int(section_budget * 0.11))),
        ChunkSpec("Rollout Edge", "Close with rollout or scaling guidance.", "Keep it practical and production-aware.", max(55, int(section_budget * 0.08))),
        ChunkSpec("Compression Latch", "Compress the implementation logic into a concise operating rule.", "Leave the next section with a clean handoff.", max(40, int(section_budget * 0.06))),
    ]

def build_longform_chunk_specs(section_budget: int) -> List[ChunkSpec]:
    return [
        ChunkSpec("Chapter Aperture", "Open with a chapter-scale framing move.", "Establish stakes, intellectual gravity, and thematic direction immediately.", max(90, int(section_budget * 0.10))),
        ChunkSpec("Thesis Weave", "State the chapter's deep promise and conceptual center.", "Explain what larger model or lens this chapter is building.", max(110, int(section_budget * 0.12))),
        ChunkSpec("Deep Context Bed", "Lay the necessary context for durable long-form understanding.", "Prefer foundational context over short-lived examples.", max(130, int(section_budget * 0.14))),
        ChunkSpec("Mechanism Chamber", "Build the core logic, system, or governing mechanism at book depth.", "Sustain explanation long enough to feel chapter-worthy.", max(170, int(section_budget * 0.18))),
        ChunkSpec("Original Framework", "Invent a useful framework, vocabulary, or model the manuscript can reuse later.", "Prefer genuinely transferable ideas over decorative branding.", max(150, int(section_budget * 0.16))),
        ChunkSpec("Counterpressure Fold", "Pressure-test the chapter against objections, edge cases, and competing views.", "Strengthen trust by letting the framework survive scrutiny.", max(120, int(section_budget * 0.12))),
        ChunkSpec("Reader Transfer Layer", "Translate the chapter into portable heuristics and operating moves.", "Make the chapter reusable outside this specific example.", max(110, int(section_budget * 0.10))),
        ChunkSpec("Chapter Bridge", "End by opening the pressure that the next chapter should resolve.", "Create manuscript momentum, not article closure.", max(80, int(section_budget * 0.08))),
    ]

def build_chunk_specs(settings: BlogSettings, section_budget: int) -> List[ChunkSpec]:
    if settings.mode == "software":
        return build_software_chunk_specs(section_budget)
    if settings.mode == "longform":
        return build_longform_chunk_specs(section_budget)
    return build_blog_chunk_specs(section_budget)

def build_upgrade_chunk_specs(dynamic_ctx: Optional[Dict[str, Any]], section_budget: int) -> List[ChunkSpec]:
    active = (dynamic_ctx or {}).get("active_upgrades") or ACTIVE_SYSTEM_UPGRADES
    specs: List[ChunkSpec] = []
    for up in active[:2]:
        specs.append(
            ChunkSpec(
                f"Upgrade Pulse: {up.get('name', 'Upgrade')}",
                up.get("principle", "Inject a refined system upgrade into the current section."),
                up.get("chunk_directive", "Use this upgrade to sharpen the current chunk."),
                max(40, int(section_budget * 0.08)),
            )
        )
    return specs

def build_micro_chunk_prompt(settings: BlogSettings, title: str, outline: List[str], section_index: int, spec: ChunkSpec, section_so_far: str, draft_so_far: str, shadow: str, next_label: str, dynamic_ctx: Optional[Dict[str, Any]] = None, surface_meta: Optional[Dict[str, Any]] = None) -> str:
    sec_title = re.sub(r"^#+\s*", "", outline[section_index]).strip()
    previous_section = re.sub(r"^#+\s*", "", outline[section_index - 1]).strip() if section_index > 0 else "(opening section)"
    next_section = re.sub(r"^#+\s*", "", outline[section_index + 1]).strip() if section_index + 1 < len(outline) else "(closing section)"
    bridge = sentence_fingerprint(section_so_far or draft_so_far)
    mode_rules = "Write like a senior engineer. Prefer APIs, flow, edge cases, and testability." if settings.mode == "software" else "Write like a serious non-fiction author building a manuscript chapter. Prefer durable conceptual depth, chapter continuity, framework recurrence, and reader transformation over article-speed closure." if settings.mode == "longform" else "Write like a strong long-form technical essayist. Prefer specificity, rhythm, and clear human explanation."
    dynamic_ctx = dynamic_ctx or {}
    surface_meta = surface_meta or {}
    retrieval_query = f"{settings.topic} {sec_title} {spec.label}"
    raw_memory_hits = retrieve_memory_surface_quantum_color(retrieval_query, topic=settings.topic, mode=settings.mode, qvec=surface_meta.get('qvec'), limit=4)
    memory_hits = render_memory_hits(raw_memory_hits)
    semantic_memory = build_semantic_compression_surface("\n".join((hit.get("summary") or "") for hit in raw_memory_hits[:4]))
    color_surface = build_colorized_thought_surface(settings.topic, "section", draft_so_far + "\n\n" + section_so_far, surface_meta.get('qvec') if surface_meta.get('qvec') is not None else quantum_state(0.5, 0.5, 0.5, section_index + 11), settings.mode, packet=ACTIVE_RESEARCH_PACKET, memory_hits=raw_memory_hits)
    active = dynamic_ctx.get("active_upgrades") or ACTIVE_SYSTEM_UPGRADES or default_refined_system_upgrades()[:4]
    upgrade_card = active[section_index % len(active)]
    invention_hint = f"{upgrade_card.get('name', 'Upgrade')}: {upgrade_card.get('principle', '')}"
    chunk_protocol = [
        "Obey the current lane sequence locally: use the first color to set pressure, the next to deepen or stabilize, and the last to hand off.",
        "Carry at least one semantic anchor forward from the bridge memory or retrieved surface.",
        "If the chromatic surface shows high compression pressure, remove one layer of unnecessary framing before adding new substance.",
        "If the chromatic surface shows high transition pressure, write the bridge sentence earlier rather than later.",
    ]
    return f"""
Document title: {title}
Overall topic: {settings.topic}
Audience: {settings.audience}
Tone: {settings.tone}
Perspective: {settings.perspective}
Mode: {settings.mode}

Current section: {sec_title}
Previous section: {previous_section}
Next section: {next_section}
Current micro-chunk: {spec.label}
Purpose: {spec.purpose}
Guidance: {spec.guidance}
Target length: about {spec.target_words} words
Forward shadow: set up the next micro-chunk called {next_label}
Stage contract: {stage_contract('section', settings.mode)}
Entropic gain: {surface_meta.get('entropic_gain', 0.0):.3f}
Chunk pressure: {surface_meta.get('chunk_pressure', 1.0):.3f}
Memory resonance: {surface_meta.get('memory_resonance', 0.0):.3f}
Invention lens: {invention_hint}
Upgrade directive: {upgrade_card.get('chunk_directive', '')}

Echo bridge memory:
{bridge or compact_tail(draft_so_far)}

Draft tail:
{compact_tail(draft_so_far, 700)}

Section tail:
{compact_tail(section_so_far, 500)}

Interstitial context:
{shadow.strip() or '(none)'}

Retrieved memory surface:
{memory_hits}

Local research packet:
{render_research_packet()}

Semantic compression surface:
{render_semantic_compression_surface(semantic_memory)}

Quantum colorized thought surface:
{render_colorized_thought_surface(color_surface)}

Advanced prompt contract:
{build_advanced_prompt_contract('section', settings.mode, color_surface)}

Chunk operating protocol:
{chr(10).join(f'- {step}' for step in chunk_protocol)}

Coherence contract:
{build_coherence_contract('section', settings.mode)}

Action protocol:
{action_protocol('section', settings.mode)}

Rules:
- Return only the prose for this micro-chunk.
- Do not restart the section from scratch.
- Do not add a heading.
- Avoid repeating phrases already used in the bridge memory.
- Make this micro-chunk feel like a necessary continuation of the previous one.
- End with semantic momentum toward the next micro-chunk, not a hard stop.
- Use one hidden [color] tag or [thought] tag when it helps keep the section on the same reasoning lane.
- Prefer a single dominant thought motion over multiple small disconnected observations.
- If you introduce abstraction, cash it out before the chunk ends unless the lane sequence explicitly reserves that payoff for the next chunk.
{mode_rules}
""".strip()

def render_section_with_invented_chunks(settings: BlogSettings, title: str, outline: List[str], section_index: int, draft_so_far: str, shadow: str = "", dynamic_ctx: Optional[Dict[str, Any]] = None, surface_meta: Optional[Dict[str, Any]] = None) -> str:
    sec_title = re.sub(r"^#+\s*", "", outline[section_index]).strip()
    section_budget = calc_section_word_budget(settings, outline, section_index)
    specs = build_chunk_specs(settings, section_budget)
    specs.extend(build_upgrade_chunk_specs(dynamic_ctx, section_budget))
    surface_meta = surface_meta or entropic_loop_metrics(900 + section_index, settings.topic, settings.mode)
    parts: List[str] = []
    for idx, spec in enumerate(specs):
        next_label = specs[idx + 1].label if idx + 1 < len(specs) else "Section Close"
        prompt = build_micro_chunk_prompt(settings, title, outline, section_index, spec, "\n\n".join(parts), draft_so_far, shadow, next_label, dynamic_ctx=dynamic_ctx, surface_meta=surface_meta)
        text = generate_chunked(prompt, max_total_new=max(80, int(spec.target_words * 1.55)), chunk_new_init=max(70, min(190, spec.target_words)), srdi_enable=True, human_style=settings.human_style_mode, memory_query=f"{settings.topic} {sec_title} {spec.label}", surface_hint=surface_meta, temperature_override=surface_meta.get('temperature'), repeat_penalty_override=surface_meta.get('repeat_penalty'), stage_name="section_chunk", mode_name=settings.mode).strip()
        if text:
            parts.append(text)
    body = "\n\n".join(parts).strip()
    if not body.lower().startswith(("##", "# ")):
        body = f"## {sec_title}\n\n{body}"
    return body

def preview_chunk_blueprint(mode: str = "blog", target_words: int = 1800, section_count: int = 6):
    settings = BlogSettings(topic="preview", mode=mode, target_words=target_words, section_count=section_count)
    outline = [f"Section {i+1}" for i in range(section_count)]
    budget = calc_section_word_budget(settings, outline, 0)
    specs = build_chunk_specs(settings, budget)
    for spec in specs:
        print(f"- {spec.label}: ~{spec.target_words} words | {spec.purpose}")

def run_blog(settings: BlogSettings, dynamic_ctx: Optional[Dict[str,Any]] = None) -> str:
    entropy, cpu, ram = get_entropy_seed()
    qvec = quantum_state(entropy, cpu, ram, 0)
    style = build_style_profile_llm(settings.topic, qvec, entropy, settings.creativity_bias)
    global ACTIVE_RESEARCH_PACKET
    global ACTIVE_COLOR_THOUGHT_SURFACE
    ACTIVE_RESEARCH_PACKET = build_research_packet_advanced(settings.topic, settings.mode, dynamic_ctx=dynamic_ctx, limit=8, qvec=qvec)
    ACTIVE_COLOR_THOUGHT_SURFACE = build_colorized_thought_surface(settings.topic, "article", "", qvec, settings.mode, packet=ACTIVE_RESEARCH_PACKET, memory_hits=ACTIVE_RESEARCH_PACKET.get("hits", []))
    agents = make_agents()
    title_consensus = run_stage_with_agents("title", agents, settings, style, None, [], None, "", 0)
    titles = parse_titles(title_consensus)
    title = settings.title_hint or (titles[0] if titles else (f"{settings.topic}: Implementation Blueprint" if settings.mode == "software" else f"{settings.topic}: A Longform Field Guide" if settings.mode == "longform" else f"{settings.topic}: A Practical Guide"))
    outline_consensus = run_stage_with_agents("outline", agents, settings, style, title, [], None, "", 1)
    outline = parse_outline(outline_consensus, settings.section_count)
    if len(outline) < settings.section_count:
        while len(outline) < settings.section_count:
            outline.append(f"Section {len(outline)+1}: Key Idea")
    spine = build_narrative_spine(title, outline, settings.topic, settings.mode)
    transition_plans = build_transition_blueprints(outline, settings.topic)
    use_forum_indices = set()
    if settings.use_forum_on_sections.strip().lower() == "all":
        use_forum_indices = set(range(len(outline)))
    elif settings.use_forum_on_sections.strip():
        try:
            use_forum_indices = set(int(x.strip()) for x in settings.use_forum_on_sections.split(",") if x.strip().isdigit())
        except Exception:
            use_forum_indices = set()
    sections_md: List[str] = []
    ledger: List[CoherenceLedgerItem] = []
    section_diagnostics: List[SectionDiagnostic] = []
    article_surface_meta = {
        "entropy": entropy,
        "cpu": cpu,
        "ram": ram,
        "qstate": format_quantum_state(qvec),
        "topic": settings.topic,
        "mode": settings.mode,
    }
    draft_so_far = f"# {title}\n\n"
    for idx, sec in enumerate(outline):
        meta = entropic_loop_metrics(2 + idx, settings.topic, settings.mode)
        shadow = ""
        if dynamic_ctx:
            shadow = build_interstitial_shadow_context(settings.topic, dynamic_ctx, 1000 + idx, meta["generators_between"], settings.creativity_bias)
        overlay = build_section_guidance_overlay(spine, transition_plans, ledger, idx, settings.mode)
        shadow = merge_shadow_with_overlay(shadow, overlay)
        qstr_now = meta["qstr"]
        section_surface_meta = dict(meta)
        section_surface_meta["coherence_overlay"] = overlay
        section_surface_meta["advanced_prompt_contract"] = build_advanced_prompt_contract('section', settings.mode, ACTIVE_COLOR_THOUGHT_SURFACE)
        section_surface_meta["chromatic_resonance"] = ACTIVE_COLOR_THOUGHT_SURFACE.get('resonance_targets', [])
        section_surface_meta["section_index"] = idx
        section_surface_meta["section_count"] = len(outline)
        section_surface_meta["section_title"] = sec
        if settings.forum_personas > 0 and (idx in use_forum_indices):
            section_prompt = f"Section target: {sec}\nAudience: {settings.audience}\nTone: {settings.tone}\nPerspective: {settings.perspective}\nCoherence overlay:\n{overlay}\nConstraints: examples, clarity, gentle humor if appropriate.\nShadow hints:\n{shadow}"
            section_text = run_peer_collab_chatlog(section_prompt=section_prompt, topic=settings.topic, qstate_str=qstr_now, entropy=meta["entropy"], personas=settings.forum_personas, n_rounds=max(1, settings.forum_rounds))
        elif ADVANCED_CHUNKING_ENABLED:
            section_text = render_section_with_invented_chunks(settings, title, outline, idx, draft_so_far, shadow=shadow, dynamic_ctx=dynamic_ctx, surface_meta=section_surface_meta)
        else:
            drift_changes = apply_quantum_style_drift((draft_so_far + "\n\n" + shadow)[-3000:], qstr_now, meta["entropy"], settings.creativity_bias)
            section_text = run_stage_with_agents("section", agents, settings, style, title, outline, idx, draft_so_far + "\n\n" + shadow, 2 + idx, drift_changes=drift_changes)
        sec_title = re.sub(r"^#+\s*", "", sec).strip()
        section_text = section_text.strip()
        if not section_text.lower().startswith(("##", "# ")):
            section_text = f"## {sec_title}\n\n{section_text}"
        previous_text = sections_md[-1] if sections_md else ""
        next_title = re.sub(r"^#+\s*", "", outline[idx + 1]).strip() if idx + 1 < len(outline) else ""
        diag = analyze_section_coherence(sec_title, section_text, previous_text=previous_text, next_title=next_title)
        repairs = build_repair_instructions(diag, next_title=next_title)
        if repairs and (diag.overall < 0.68 or len(diag.issues) >= 2):
            section_text = run_targeted_section_repair(sec_title, section_text, repairs, next_title, settings, section_surface_meta)
            diag = analyze_section_coherence(sec_title, section_text, previous_text=previous_text, next_title=next_title)
        sections_md.append(section_text)
        section_diagnostics.append(diag)
        ledger.append(ledger_item_from_section(sec_title, section_text, diag))
        store_memory_surface(section_text, stage="section", topic=settings.topic, mode=settings.mode, title=title, salience=max(0.72, min(0.94, diag.overall)))
        store_memory_surface(json.dumps(asdict(diag), ensure_ascii=False), stage="section_diagnostic", topic=settings.topic, mode=settings.mode, title=title, salience=max(0.70, min(0.9, diag.overall)))
        draft_so_far += section_text + "\n\n"
    store_coherence_ledger(settings.topic, settings.mode, ledger)
    article_diag_pre = analyze_article_coherence(title, sections_md, ledger)
    store_article_diagnostic(settings.topic, settings.mode, article_diag_pre)
    store_memory_surface(render_article_diagnostic(article_diag_pre), stage="article_diagnostic_report", topic=settings.topic, mode=settings.mode, title=title, salience=max(0.72, min(0.92, article_diag_pre.overall)))
    final_consensus = run_stage_with_agents("polish", agents, settings, style, title, outline, None, draft_so_far, 2 + len(outline))
    polished_body = extract_article_body(final_consensus, title)
    article_body = polished_body or "\n\n".join(sections_md).strip()
    if not re.search(r"(?im)^##\s+(conclusion|wrap\-up|final thoughts)\b", article_body):
        article_body = (article_body + "\n\n" if article_body else "") + "## Conclusion\n\n" + textwrap.dedent("""\
            The short version? Pick one practical step and try it. Adjust as you go.
            The rest will make more sense when it's in motion, not on paper.""").strip()
    article_body = run_article_coherence_repair(title, article_body, article_diag_pre, ledger, settings, article_surface_meta)
    article_sections_for_diag = [chunk.strip() for chunk in re.split(r"(?m)(?=^##\s+)", article_body) if chunk.strip()]
    article_diag_post = analyze_article_coherence(title, article_sections_for_diag or sections_md, ledger)
    store_article_diagnostic(settings.topic, settings.mode, article_diag_post)
    meta_source = article_body or polished_body or final_consensus
    meta_description = ""
    if settings.include_meta_description:
        words = re.findall(r"\w[\w\-']*", meta_source)
        meta_description = " ".join(words[:28]) + ("..." if len(words) > 28 else "")
    front_matter = ""
    if settings.include_front_matter:
        fm = {
            "title": title,
            "description": meta_description.strip(),
            "author": settings.author_name or "Staff Writer",
            "keywords": settings.keywords or "",
            "readingGrade": settings.reading_grade,
        }
        fm_lines = ["---"] + [f"{k}: {v}" for k, v in fm.items()] + ["---", ""]
        front_matter = "\n".join(fm_lines)
    final_md = []
    if front_matter:
        final_md.append(front_matter)
    final_md.append(f"# {title}\n")
    if settings.include_meta_description and meta_description:
        final_md.append(f"> {meta_description}\n")
    final_md.append(article_body)
    final_article = "\n\n".join(part.strip() for part in final_md if part and part.strip()).strip()
    if settings.ghostwriter_passes and settings.ghostwriter_passes > 0:
        qstr_end = format_quantum_state(quantum_state(entropy, cpu, ram, 7))
        final_article = run_adversarial_ghostwriter_chain(final_article, settings.topic, qstr_end, entropy, passes=settings.ghostwriter_passes)
    store_memory_surface(final_article, stage="final_article", topic=settings.topic, mode=settings.mode, title=title, salience=1.0)
    update_knowledge_graph(settings.topic, final_article)
    chosen_policy = choose_prompt_policy(settings.topic, settings.mode, "final_article")
    prompt_notes = render_active_upgrades(field="name") + " | coherence=" + ", ".join(article_diag_post.issues or ["stable"])
    record_prompt_experiment(stage="final_article", mode=settings.mode, topic=settings.topic, prompt_text=settings.root_prompt + "\n" + chosen_policy.directive, score=score_article_surface(final_article), notes=prompt_notes)
    return final_article

def run_longform_manuscript(settings: BlogSettings, dynamic_ctx: Optional[Dict[str, Any]] = None) -> str:
    entropy, cpu, ram = get_entropy_seed()
    qvec = quantum_state(entropy, cpu, ram, 21)
    style = build_style_profile_llm(settings.topic, qvec, entropy, settings.creativity_bias)
    agents = make_agents()
    chapter_count = max(6, settings.section_count)
    title_consensus = run_stage_with_agents("title", agents, settings, style, None, [], None, "", 0)
    titles = parse_titles(title_consensus)
    manuscript_title = settings.title_hint or (titles[0] if titles else f"{settings.topic}: A Longform Field Guide")
    outline_consensus = run_stage_with_agents("outline", agents, settings, style, manuscript_title, [], None, "", 1)
    chapter_outline = parse_outline(outline_consensus, chapter_count)
    while len(chapter_outline) < chapter_count:
        chapter_outline.append(f"## Chapter {len(chapter_outline)+1}: Core Idea")
    dynamic_ctx = dict(dynamic_ctx or {})
    dynamic_ctx["manuscript_title"] = manuscript_title
    dynamic_ctx["chapter_outline"] = [re.sub(r"^#+\s*", "", x).strip() for x in chapter_outline]
    dynamic_ctx["book_mode"] = True
    chapter_target_words = max(2200, int(max(settings.target_words, chapter_count * 2200) / max(1, chapter_count)))
    chapter_section_count = max(5, min(9, 6 if chapter_target_words < 3200 else 7 if chapter_target_words < 5200 else 8))
    chapter_outputs: List[str] = []
    chapter_summaries: List[str] = []
    for idx, chapter in enumerate(chapter_outline):
        chapter_title = re.sub(r"^#+\s*", "", chapter).strip()
        chapter_map_text = "\n".join("- " + re.sub(r"^#+\s*", "", x).strip() for x in chapter_outline)
        prior_summaries_text = "\n".join("- " + s for s in chapter_summaries[-4:]) if chapter_summaries else "(none yet)"
        chapter_context = textwrap.dedent(f"""
        Manuscript title: {manuscript_title}
        Chapter number: {idx + 1}/{len(chapter_outline)}
        Current chapter: {chapter_title}
        Full chapter map:
        {chapter_map_text}
        Prior chapter summaries:
        {prior_summaries_text}
        Write this as part of a book-length manuscript. Favor continuity, recurrence of frameworks, and chapter-scale payoff.
        """).strip()
        chapter_dynamic_ctx = dict(dynamic_ctx)
        chapter_dynamic_ctx.setdefault("retrieval_queries", [])
        chapter_dynamic_ctx["retrieval_queries"] = dedupe_preserve_order(chapter_dynamic_ctx["retrieval_queries"] + [chapter_title, manuscript_title, f"{settings.topic} {chapter_title} framework"])
        chapter_dynamic_ctx["prior_chapter_summaries"] = list(chapter_summaries[-6:])
        chapter_settings = BlogSettings(
            topic=f"{settings.topic} / {chapter_title}",
            mode="longform",
            root_prompt=settings.root_prompt + "\n\n" + chapter_context,
            audience=settings.audience,
            tone=settings.tone,
            perspective=settings.perspective,
            reading_grade=settings.reading_grade,
            target_words=chapter_target_words,
            section_count=chapter_section_count,
            human_style_mode=settings.human_style_mode,
            title_hint=f"Chapter {idx + 1}: {chapter_title}",
            keywords=settings.keywords,
            author_name=settings.author_name,
            include_front_matter=False,
            include_meta_description=False,
            forum_personas=settings.forum_personas,
            forum_rounds=settings.forum_rounds,
            ghostwriter_passes=max(0, settings.ghostwriter_passes - 1),
            creativity_bias=settings.creativity_bias,
            use_forum_on_sections=settings.use_forum_on_sections,
        )
        chapter_md = run_blog(chapter_settings, dynamic_ctx=chapter_dynamic_ctx)
        chapter_outputs.append(chapter_md)
        chapter_body = extract_article_body(chapter_md, f"Chapter {idx + 1}: {chapter_title}")
        chapter_summary = short_text_summary(chapter_body, max_sentences=4, max_chars=420)
        chapter_summaries.append(chapter_summary)
        store_memory_surface(chapter_summary, stage="chapter_summary", topic=settings.topic, mode="longform", title=manuscript_title, salience=0.92)
    pages_estimate = max(1, int(max(settings.target_words, chapter_count * chapter_target_words) / 300))
    cleaned_chapter_titles = [re.sub(r"^#+\s*", "", x).strip() for x in chapter_outline]
    toc = "\n".join(f"- Chapter {i+1}: {x}" for i, x in enumerate(cleaned_chapter_titles))
    manuscript_parts = [f"# {manuscript_title}", "## Table of Contents\n" + toc, "## Manuscript Notes\n" + f"Approximate pages at 300 words/page: {pages_estimate}\n\nTotal target words: {max(settings.target_words, chapter_count * chapter_target_words)}"]
    manuscript_parts.extend(chapter_outputs)
    manuscript = "\n\n".join(part.strip() for part in manuscript_parts if part and part.strip()).strip()
    store_memory_surface(manuscript, stage="final_manuscript", topic=settings.topic, mode="longform", title=manuscript_title, salience=1.0)
    update_knowledge_graph(settings.topic, manuscript)
    record_prompt_experiment(stage="final_manuscript", mode="longform", topic=settings.topic, prompt_text=settings.root_prompt, score=score_article_surface(manuscript), notes=f"chapters={chapter_count} | chapter_target_words={chapter_target_words}")
    return manuscript

def autonomous_factory_scheduler(seed_topics: List[str], mode: str = "blog", batch_size: int = 3) -> List[Dict[str, Any]]:
    expanded = semantic_topic_expansion(seed_topics, mode, top_k=batch_size * 4, threshold=0.18)
    pool = dedupe_preserve_order(seed_topics + expanded)
    jobs = []
    for idx, topic in enumerate(pool[:batch_size]):
        neighbors = graph_neighbors(topic, limit=5)
        jobs.append({
            "topic": topic,
            "mode": mode,
            "priority": round(1.0 - 0.08 * idx, 3),
            "policy": choose_prompt_policy(topic, mode, "section").name,
            "neighbors": neighbors,
        })
    return jobs

def autonomous_blog_factory(seed_topics: List[str], mode: str = "blog", max_articles: int = 3) -> List[Dict[str, str]]:
    ideas = dedupe_preserve_order(seed_topics + semantic_topic_expansion(seed_topics, mode, top_k=max_articles * 3, threshold=0.18))
    umbrella = f"{mode} topics"
    dynamic_ctx = build_origin_dynamic_context(umbrella, ideas[:max_articles], "general readers", "clear, human, practical", "direct expert guidance", "", 0.95)
    scheduler = autonomous_factory_scheduler(ideas[:max_articles], mode=mode, batch_size=max_articles)
    outputs = []
    for idx, job in enumerate(scheduler):
        topic = job["topic"]
        settings = BlogSettings(topic=topic, mode=mode, root_prompt=default_root_prompt_for_mode(mode), audience="general readers", tone="clear, human, practical", perspective="direct expert guidance", target_words=default_target_words_for_mode(mode), section_count=default_section_count_for_mode(mode))
        article = run_longform_manuscript(settings, dynamic_ctx=dynamic_ctx) if mode == "longform" else run_blog(settings, dynamic_ctx=dynamic_ctx)
        outputs.append({"topic": topic, "article": article, "policy": job["policy"], "priority": str(job["priority"]), "neighbors": ", ".join(job["neighbors"])})
    return outputs

# =========================
# Entrypoints
# =========================
def main_colab():
    mode = normalize_mode(str(STANDARD_INPUTS["mode"]))
    topics = normalize_topics(STANDARD_INPUTS["topics"], mode)
    audience = STANDARD_INPUTS["audience_hint"].strip() or "general readers"
    tone = STANDARD_INPUTS["tone_hint"].strip() or "clear, human, practical"
    perspective = STANDARD_INPUTS["perspective_hint"].strip() or "direct expert guidance"
    keywords = STANDARD_INPUTS["keywords_hint"].strip()
    include_fm = bool(STANDARD_INPUTS["include_front_matter"])
    include_meta = bool(STANDARD_INPUTS["include_meta_description"])
    creativity = float(STANDARD_INPUTS["creativity"])
    count = max(1, int(STANDARD_INPUTS["count"]))
    outfile = STANDARD_INPUTS["outfile"].strip() or default_outfile_for_mode(mode)
    root_prompt = STANDARD_INPUTS["root_prompt"].strip() or default_root_prompt_for_mode(mode)

    umbrella = f"{mode} topics"
    entropy, cpu, ram = get_entropy_seed()
    qvec = quantum_state(entropy, cpu, ram, 99)
    dynamic_ctx = build_origin_dynamic_context(umbrella, topics, audience, tone, perspective, keywords, creativity)

    expanded_topics = semantic_topic_expansion(topics, mode, top_k=max(4, count * 2), threshold=0.18)
    topics_for_run = dedupe_preserve_order(topics + expanded_topics)[:count]
    needed = max(0, count - len(topics_for_run))
    if needed > 0:
        new_ideas = llm_generate_new_ideas(umbrella, topics_for_run, qvec, entropy, n=needed)
        topics_for_run = dedupe_preserve_order(topics_for_run + new_ideas)
    if not topics_for_run:
        topics_for_run = list(DEFAULT_TOPIC_BANK.get(mode, DEFAULT_TOPIC_BANK["blog"]))[:count]
    topics_for_run = topics_for_run[:count]

    outputs = []
    for i, t in enumerate(topics_for_run, start=1):
        settings = BlogSettings(
            topic=t,
            mode=mode,
            root_prompt=root_prompt,
            audience=audience,
            tone=tone,
            perspective=perspective,
            reading_grade="8-10",
            target_words=max(600, int(STANDARD_INPUTS["words"])),
            section_count=max(3, int(STANDARD_INPUTS["sections"])),
            human_style_mode=True,
            title_hint=None,
            keywords=keywords,
            author_name=None,
            include_front_matter=include_fm,
            include_meta_description=include_meta,
            forum_personas=max(0, int(STANDARD_INPUTS["forum_personas"])),
            forum_rounds=max(0, int(STANDARD_INPUTS["forum_rounds"])),
            ghostwriter_passes=max(0, int(STANDARD_INPUTS["ghostwriter_passes"])),
            creativity_bias=max(0.1, min(1.5, creativity)),
            use_forum_on_sections=str(STANDARD_INPUTS["forum_on"]).strip(),
        )
        article = run_longform_manuscript(settings, dynamic_ctx=dynamic_ctx) if mode == "longform" else run_blog(settings, dynamic_ctx=dynamic_ctx)
        if count == 1:
            out_path = outfile
        else:
            root, ext = (outfile, "") if "." not in outfile else (outfile.rsplit(".", 1)[0], "." + outfile.rsplit(".",1)[1])
            out_path = f"{root}_{i}{ext or '.md'}"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(article + "\n")
        outputs.append((t, out_path))

    manifest = {
        "mode": mode,
        "base_topic": umbrella,
        "count": count,
        "seed_ideas": topics,
        "generated_topics": topics_for_run[:count],
        "outputs": [{"topic": t, "path": p} for (t, p) in outputs],
        "settings": {
            "audience": audience,
            "tone": tone,
            "perspective": perspective,
            "grade": "8-10",
            "words": int(STANDARD_INPUTS["words"]),
            "sections": int(STANDARD_INPUTS["sections"]),
            "forum_personas": int(STANDARD_INPUTS["forum_personas"]),
            "forum_rounds": int(STANDARD_INPUTS["forum_rounds"]),
            "forum_on": str(STANDARD_INPUTS["forum_on"]),
            "ghostwriter_passes": int(STANDARD_INPUTS["ghostwriter_passes"]),
            "creativity": creativity,
            "root_prompt": root_prompt,
        }
    }
    with open("run_manifest.json", "w", encoding="utf-8") as mf:
        json.dump(manifest, mf, ensure_ascii=False, indent=2)

def main_cli():
    parser = argparse.ArgumentParser()
    parser.add_argument("--topic", type=str, required=True)
    parser.add_argument("--mode", type=str, default="blog", choices=["blog", "software", "longform"])
    parser.add_argument("--root-prompt", type=str, default="")
    parser.add_argument("--ideas", type=str, default="")
    parser.add_argument("--count", type=int, default=1)
    parser.add_argument("--audience", type=str, default="general readers")
    parser.add_argument("--tone", type=str, default="warm, practical, lightly witty")
    parser.add_argument("--perspective", type=str, default="second-person 'you' with occasional 'we'")
    parser.add_argument("--grade", type=str, default="8-10")
    parser.add_argument("--words", type=int, default=0)
    parser.add_argument("--sections", type=int, default=0)
    parser.add_argument("--title_hint", type=str, default=None)
    parser.add_argument("--keywords", type=str, default=None)
    parser.add_argument("--author", type=str, default=None)
    parser.add_argument("--front_matter", action="store_true")
    parser.add_argument("--no_meta_desc", action="store_true")
    parser.add_argument("--outfile", type=str, default="")
    parser.add_argument("--forum_personas", type=int, default=0)
    parser.add_argument("--forum_rounds", type=int, default=0)
    parser.add_argument("--forum_on", type=str, default="")
    parser.add_argument("--ghostwriter_passes", type=int, default=0)
    parser.add_argument("--creativity", type=float, default=0.9)
    args = parser.parse_args()

    seed_ideas = []
    if args.ideas:
        parts = re.split(r"[;\n]", args.ideas)
        seed_ideas = [p.strip() for p in parts if p.strip()]

    entropy, cpu, ram = get_entropy_seed()
    qvec = quantum_state(entropy, cpu, ram, 99)
    dyn = build_origin_dynamic_context(args.topic, seed_ideas, args.audience, args.tone, args.perspective, args.keywords or "", args.creativity)

    topics_for_run: List[str] = []
    topics_for_run.extend(seed_ideas[:max(0, args.count)])
    needed = max(0, args.count - len(topics_for_run))
    if needed > 0:
        new_ideas = llm_generate_new_ideas(args.topic, seed_ideas, qvec, entropy, n=needed)
        topics_for_run.extend(new_ideas)
    if not topics_for_run:
        topics_for_run = [f"{args.topic} — Angle {i+1}" for i in range(args.count)]

    outputs: List[Tuple[str, str]] = []
    for i, t in enumerate(topics_for_run[:args.count], start=1):
        settings = BlogSettings(
            topic=t,
            mode=args.mode,
            root_prompt=args.root_prompt.strip() or default_root_prompt_for_mode(args.mode),
            audience=args.audience,
            tone=args.tone,
            perspective=args.perspective,
            reading_grade=args.grade,
            target_words=max(600, args.words or default_target_words_for_mode(args.mode)),
            section_count=max(3, args.sections or default_section_count_for_mode(args.mode)),
            human_style_mode=True,
            title_hint=args.title_hint,
            keywords=args.keywords,
            author_name=args.author,
            include_front_matter=bool(args.front_matter),
            include_meta_description=not args.no_meta_desc,
            forum_personas=max(0, args.forum_personas),
            forum_rounds=max(0, args.forum_rounds),
            ghostwriter_passes=max(0, args.ghostwriter_passes),
            creativity_bias=max(0.1, min(1.5, args.creativity)),
            use_forum_on_sections=args.forum_on.strip(),
        )
        article = run_longform_manuscript(settings, dynamic_ctx=dyn) if args.mode == "longform" else run_blog(settings, dynamic_ctx=dyn)
        if args.count == 1:
            out_path = args.outfile or default_outfile_for_mode(args.mode)
        else:
            base_out = args.outfile or default_outfile_for_mode(args.mode)
            root, ext = (base_out, "") if "." not in base_out else (base_out.rsplit(".", 1)[0], "." + base_out.rsplit(".",1)[1])
            out_path = f"{root}_{i}{ext or '.md'}"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(article + "\n")
        outputs.append((t, out_path))

    manifest = {
        "base_topic": args.topic,
        "count": args.count,
        "seed_ideas": seed_ideas,
        "generated_topics": topics_for_run[:args.count],
        "outputs": [{"topic": t, "path": p} for (t, p) in outputs],
        "settings": {
            "audience": args.audience,
            "tone": args.tone,
            "perspective": args.perspective,
            "grade": args.grade,
            "words": args.words,
            "sections": args.sections,
            "forum_personas": args.forum_personas,
            "forum_rounds": args.forum_rounds,
            "forum_on": args.forum_on,
            "ghostwriter_passes": args.ghostwriter_passes,
            "creativity": args.creativity,
        }
    }
    with open("run_manifest.json", "w", encoding="utf-8") as mf:
        json.dump(manifest, mf, ensure_ascii=False, indent=2)

if __name__ == "__main__":
    if os.environ.get("USE_COLAB_INPUTS", "1") == "1":
        main_colab()
    else:
        main_cli()